# PonziMamba — Deployment-Time Smart Ponzi Detection with a State-Space Model

A simplified **Mamba (selective SSM)** classifier over tokenized Solidity source code,
with **focal loss** for class imbalance, evaluated on a **strictly deduplicated** version
of the standard XBlock benchmark (6,498 contracts). Detection uses only contract code
(no transaction history), so it works the moment a contract is verified — addressing the
early-detection gap in prior work.

**Benchmark audit built into this notebook** (a paper finding on its own):
581 exact duplicates + 14,648 near-duplicate pairs (MinHash-LSH, Jaccard ≥ 0.9),
including 113 pairs with **conflicting labels** → strict benchmark of 4,021 contracts (158 Ponzi).
XGBoost baseline drops from F1 0.82 (exact-dedup only) to **0.64 ± 0.06** (strict),
showing prior results are inflated by train/test leakage.

**Reviewer checklist addressed:** Transformer baseline of the same size class ·
threshold tuned on validation only (0.5 also reported) · stratified 5-fold CV with
per-fold confusion matrices · exact + near-duplicate leakage control with released
pair list · efficiency measurements (params, tokens/sec, peak GPU memory) ·
seeds + library versions logged per run.

Run the cells top-to-bottom. Results accumulate in `results.jsonl` — send that file back for analysis.

## 0 · Setup

In [1]:
# Uncomment on first run (Colab: also Runtime > Change runtime type > GPU)
# %pip install -q torch xgboost scikit-learn numpy pandas

In [2]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("!" * 66)
    print("! WARNING: torch cannot see a GPU. All deep-model cells will be")
    print("! EXTREMELY slow. On Windows, plain `pip install torch` gives a")
    print("! CPU-only build. If this machine has an NVIDIA GPU, run:")
    print("!   pip install -U torch --index-url https://download.pytorch.org/whl/cu126")
    print("! then RESTART the kernel and re-run this cell until it prints")
    print("! your GPU name.")
    print("!" * 66)

torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 1 · Data preparation
Clones the public [SourceP](https://github.com/Lpover/SourceP) release of the XBlock
ground truth (~110 MB, once), merges official labels, removes exact duplicates, then finds
and removes near-duplicates via MinHash-LSH. Produces `data/ponzi_clean.jsonl` and the
audit file `data/near_duplicate_pairs.csv`. Skipped automatically if the data already exists.
Set `args_prep.keep_near_dups = True` to build the leaky (exact-dedup-only) variant instead.

In [3]:
import argparse
import hashlib
import json
import os
import re
import subprocess
import sys
from collections import defaultdict

REPO_URL = "https://github.com/Lpover/SourceP.git"
REPO_DIR = "SourceP"
OUT_PATH = os.path.join("data", "ponzi_clean.jsonl")
NEAR_DUP_PATH = os.path.join("data", "near_duplicate_pairs.csv")
LABEL_FILES = ["train.txt", "valid.txt", os.path.join("RQ1", "dataset", "test.txt")]

WORD_RE = re.compile(r"[A-Za-z_$][A-Za-z0-9_$]*|[^\sA-Za-z0-9_$]")


def shingles(src: str, k: int = 5):
    toks = WORD_RE.findall(src)
    return {hash(tuple(toks[i:i + k])) for i in range(max(len(toks) - k + 1, 1))}


def find_near_duplicates(records, jaccard_thr=0.9, n_hash=64, bands=16):
    """MinHash-LSH candidate generation + exact Jaccard verification.
    Returns list of (i, j, jaccard) pairs with jaccard >= threshold."""
    import random as _random
    rng = _random.Random(0)
    masks = [rng.getrandbits(64) for _ in range(n_hash)]
    shingle_sets, signatures = [], []
    for r in records:
        s = shingles(r["sourcecode"])
        shingle_sets.append(s)
        sig = tuple(min((h ^ m) & 0xFFFFFFFFFFFFFFFF for h in s)
                    for m in masks)
        signatures.append(sig)

    rows = n_hash // bands
    candidates = set()
    for b in range(bands):
        buckets = defaultdict(list)
        for i, sig in enumerate(signatures):
            buckets[sig[b * rows:(b + 1) * rows]].append(i)
        for members in buckets.values():
            if len(members) > 1:
                for a in range(len(members)):
                    for c in range(a + 1, len(members)):
                        candidates.add((members[a], members[c]))

    pairs = []
    for i, j in candidates:
        si, sj = shingle_sets[i], shingle_sets[j]
        jac = len(si & sj) / max(len(si | sj), 1)
        if jac >= jaccard_thr:
            pairs.append((i, j, jac))
    return pairs


def main() -> None:
    if not os.path.isdir(REPO_DIR):
        # pure-Python download (no git required - works on Windows)
        import urllib.request
        import zipfile
        import shutil
        url = "https://github.com/Lpover/SourceP/archive/refs/heads/main.zip"
        zip_path = "SourceP_repo.zip"
        print(f"Downloading {url} (~110 MB)...")
        urllib.request.urlretrieve(url, zip_path)
        print("Extracting...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(".")
        shutil.move("SourceP-main", REPO_DIR)
        os.remove(zip_path)

    ds_dir = os.path.join(REPO_DIR, "dataset")

    labels = {}
    for fn in LABEL_FILES:
        path = os.path.join(ds_dir, fn)
        with open(path) as f:
            for line in f:
                parts = line.split()
                if len(parts) == 2:
                    labels[parts[0]] = int(parts[1])
    print(f"Labels loaded: {len(labels)} contracts, "
          f"{sum(labels.values())} Ponzi")

    os.makedirs("data", exist_ok=True)
    seen, dups, records = set(), 0, []
    with open(os.path.join(ds_dir, "data.jsonl")) as fin:
        for line in fin:
            d = json.loads(line)
            idx = str(d["idx"])
            if idx not in labels:
                continue
            src = d["sourcecode"]
            digest = hashlib.md5(re.sub(r"\s+", "", src).encode()).hexdigest()
            if digest in seen:
                dups += 1
                continue
            seen.add(digest)
            records.append({
                "idx": idx,
                "address": d.get("address", ""),
                "label": labels[idx],
                "sourcecode": src,
            })
    print(f"Exact duplicates removed: {dups}  (kept {len(records)})")

    # ---- near-duplicate detection (MinHash-LSH + Jaccard >= threshold) ----
    print(f"Scanning for near-duplicates (Jaccard >= {args.jaccard}) ...")
    pairs = find_near_duplicates(records, jaccard_thr=args.jaccard)
    with open(NEAR_DUP_PATH, "w") as f:
        f.write("idx_a,idx_b,label_a,label_b,jaccard\n")
        for i, j, jac in sorted(pairs, key=lambda p: -p[2]):
            f.write(f"{records[i]['idx']},{records[j]['idx']},"
                    f"{records[i]['label']},{records[j]['label']},"
                    f"{jac:.3f}\n")
    cross_label = sum(1 for i, j, _ in pairs
                      if records[i]["label"] != records[j]["label"])
    print(f"Near-duplicate pairs found: {len(pairs)} "
          f"({cross_label} with conflicting labels) -> {NEAR_DUP_PATH}")

    if not args.keep_near_dups and pairs:
        # union-find: keep one representative per near-duplicate group
        parent = list(range(len(records)))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for i, j, _ in pairs:
            parent[find(i)] = find(j)
        groups = defaultdict(list)
        for i in range(len(records)):
            groups[find(i)].append(i)
        keep_idx = set()
        for members in groups.values():
            # prefer keeping a Ponzi sample as the representative
            members.sort(key=lambda i: (-records[i]["label"],
                                        records[i]["idx"]))
            keep_idx.add(members[0])
        removed = len(records) - len(keep_idx)
        records = [r for i, r in enumerate(records) if i in keep_idx]
        print(f"Near-duplicates removed: {removed} "
              f"(1 representative kept per group)")

    ponzi = sum(r["label"] for r in records)
    with open(OUT_PATH, "w") as fout:
        for rec in records:
            fout.write(json.dumps(rec) + "\n")
    print(f"Final dataset: {len(records)} contracts ({ponzi} Ponzi, "
          f"{len(records) - ponzi} non-Ponzi) -> {OUT_PATH}")

In [4]:
from types import SimpleNamespace
import os
args = args_prep = SimpleNamespace(jaccard=0.9, keep_near_dups=False)
if os.path.exists(OUT_PATH):
    print(f"{OUT_PATH} already exists - skipping preparation. Delete it to rebuild.")
else:
    main()

data\ponzi_clean.jsonl already exists - skipping preparation. Delete it to rebuild.


## 2 · Models
Pure PyTorch — no custom CUDA kernels, runs on any GPU/CPU:
`MambaClassifier` (simplified selective SSM / S6), `TransformerClassifier`
(attention counterpart, same size class), `TextCNNClassifier`, and binary `FocalLoss`.

In [5]:
"""
models.py
Pure-PyTorch implementations (no custom CUDA kernels required):

- MambaClassifier : simplified Mamba / selective state-space model (S6)
                    over token sequences. Runs on any GPU or CPU.
- TextCNNClassifier : multi-kernel TextCNN baseline (Kim, 2014 style),
                      close in spirit to prior Ponzi-detection CNNs.
- FocalLoss : binary focal loss for class imbalance (Lin et al., 2017).
"""

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------------
# Focal loss
# --------------------------------------------------------------------------
class FocalLoss(nn.Module):
    """Binary focal loss on logits. alpha weights the positive (Ponzi) class."""

    def __init__(self, alpha: float = 0.75, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.float()
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - p_t).pow(self.gamma) * bce
        return loss.mean()


# --------------------------------------------------------------------------
# Simplified Mamba block (selective SSM, sequential scan)
# --------------------------------------------------------------------------
FORCE_PARALLEL_SCAN = False  # test hook: force GPU scan path on CPU
_SCAN_CHUNK = 128             # chunk length for the memory-bounded GPU scan


def _chunk_scan(delta_c, xc_c, Bmat_c, Cmat_c, carry, A):
    """Selective-SSM scan over ONE chunk of the sequence.
    Computes discretized (dA, dBx) locally, runs a Hillis-Steele parallel
    scan (log2(chunk) tensor ops), folds in the carry state from previous
    chunks, and immediately contracts the hidden states with C so the big
    (B, chunk, d_inner, d_state) tensors never leave this function.
    Designed to be wrapped in gradient checkpointing: only the small 3-D
    inputs are stored; everything here is recomputed in backward, one
    chunk at a time, keeping peak GPU memory flat in sequence length."""
    dA = torch.exp(delta_c.unsqueeze(-1) * A.view(1, 1, *A.shape))
    dBx = (delta_c * xc_c).unsqueeze(-1) * Bmat_c.unsqueeze(2)
    a, b = dA, dBx
    Lc = a.shape[1]
    d = 1
    while d < Lc:
        b = torch.cat([b[:, :d], b[:, d:] + a[:, d:] * b[:, :-d]], dim=1)
        a = torch.cat([a[:, :d], a[:, d:] * a[:, :-d]], dim=1)
        d *= 2
    h = b + a * carry.unsqueeze(1)
    y = torch.einsum("bldn,bln->bld", h, Cmat_c)
    return y, h[:, -1]


class MambaBlock(nn.Module):
    """
    Simplified Mamba block:
      in_proj -> (x, z); causal depthwise conv on x; SiLU;
      selective SSM with input-dependent (delta, B, C); gate by SiLU(z);
      out_proj.
    The SSM recurrence uses a parallel scan (log2(L) large tensor ops)
    wrapped in gradient checkpointing to keep memory flat; on CPU a
    sequential scan is used instead (fewer memory passes wins there).
    """

    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4,
                 expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = expand * d_model
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            padding=d_conv - 1, groups=self.d_inner,
        )
        # x -> delta (1), B (d_state), C (d_state)
        self.x_proj = nn.Linear(self.d_inner, 1 + 2 * d_state, bias=False)
        self.dt_bias = nn.Parameter(torch.zeros(self.d_inner))

        # S4D-real initialization of A (negative real)
        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A).repeat(self.d_inner, 1))
        self.D = nn.Parameter(torch.ones(self.d_inner))

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # (B, L, D)
        B_, L, _ = x.shape
        xz = self.in_proj(x)                       # (B, L, 2*d_inner)
        x_in, z = xz.chunk(2, dim=-1)

        x_c = self.conv1d(x_in.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x_c = F.silu(x_c)                          # (B, L, d_inner)

        dbc = self.x_proj(x_c)                     # (B, L, 1+2*d_state)
        delta, Bmat, Cmat = torch.split(
            dbc, [1, self.d_state, self.d_state], dim=-1)
        # broadcast scalar delta across channels, add learned per-channel bias
        delta = F.softplus(delta.expand(-1, -1, self.d_inner)
                           + self.dt_bias.view(1, 1, -1))   # (B, L, d_inner)

        A = -torch.exp(self.A_log)                 # (d_inner, d_state)

        if x.device.type == "cpu" and not FORCE_PARALLEL_SCAN:
            # sequential scan: fine on CPU (no kernel-launch overhead)
            dA_all = torch.exp(delta.unsqueeze(-1) * A.view(
                1, 1, self.d_inner, self.d_state))
            dBx_all = (delta * x_c).unsqueeze(-1) * Bmat.unsqueeze(2)
            h = torch.zeros(B_, self.d_inner, self.d_state,
                            device=x.device, dtype=x.dtype)
            ys = []
            for t in range(L):
                h = dA_all[:, t] * h + dBx_all[:, t]
                ys.append(h)
            h_all = torch.stack(ys, dim=1)
            y = torch.einsum("bldn,bln->bld", h_all, Cmat)
        else:
            # chunked parallel scan with per-chunk gradient checkpointing:
            # peak GPU memory stays flat regardless of sequence length
            carry = torch.zeros(B_, self.d_inner, self.d_state,
                                device=x.device, dtype=x.dtype)
            ys = []
            use_ckpt = self.training and torch.is_grad_enabled()
            for s in range(0, L, _SCAN_CHUNK):
                e = min(s + _SCAN_CHUNK, L)
                chunk_args = (delta[:, s:e], x_c[:, s:e], Bmat[:, s:e],
                              Cmat[:, s:e], carry, A)
                if use_ckpt:
                    y_c, carry = torch.utils.checkpoint.checkpoint(
                        _chunk_scan, *chunk_args,
                        use_reentrant=False, preserve_rng_state=False)
                else:
                    y_c, carry = _chunk_scan(*chunk_args)
                ys.append(y_c)
            y = torch.cat(ys, dim=1)
        y = y + self.D.view(1, 1, -1) * x_c
        y = y * F.silu(z)
        return self.out_proj(y)


class MambaClassifier(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 128, n_layers: int = 2,
                 d_state: int = 16, dropout: float = 0.2, pad_id: int = 0):
        super().__init__()
        self.pad_id = pad_id
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.blocks = nn.ModuleList([MambaBlock(d_model, d_state)
                                     for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model)
                                    for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, 1)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:  # (B, L)
        mask = (ids != self.pad_id).float().unsqueeze(-1)   # (B, L, 1)
        x = self.embed(ids)
        for norm, block in zip(self.norms, self.blocks):
            x = x + block(norm(x))
        x = self.final_norm(x)
        x = (x * mask).sum(1) / mask.sum(1).clamp(min=1.0)  # masked mean-pool
        return self.head(self.dropout(x)).squeeze(-1)


# --------------------------------------------------------------------------
# Transformer encoder baseline (attention counterpart to the SSM)
# --------------------------------------------------------------------------
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 128, n_layers: int = 2,
                 n_heads: int = 4, max_len: int = 2048, dropout: float = 0.2,
                 pad_id: int = 0):
        super().__init__()
        self.pad_id = pad_id
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=4 * d_model,
            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, 1)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:  # (B, L)
        pad_mask = ids == self.pad_id                       # (B, L)
        positions = torch.arange(ids.size(1), device=ids.device)
        x = self.embed(ids) + self.pos(positions).unsqueeze(0)
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        x = self.final_norm(x)
        mask = (~pad_mask).float().unsqueeze(-1)
        x = (x * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        return self.head(self.dropout(x)).squeeze(-1)


# --------------------------------------------------------------------------
# TextCNN baseline
# --------------------------------------------------------------------------
class TextCNNClassifier(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 128,
                 kernel_sizes=(3, 5, 7), n_filters: int = 100,
                 dropout: float = 0.3, pad_id: int = 0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.convs = nn.ModuleList(
            [nn.Conv1d(d_model, n_filters, k, padding=k // 2)
             for k in kernel_sizes])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(n_filters * len(kernel_sizes), 1)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        x = self.embed(ids).transpose(1, 2)        # (B, D, L)
        feats = [F.relu(conv(x)).max(dim=2).values for conv in self.convs]
        x = torch.cat(feats, dim=1)
        return self.head(self.dropout(x)).squeeze(-1)


# --------------------------------------------------------------------------
# v2: Bidirectional Mamba + attention pooling + hybrid lexical branch
# --------------------------------------------------------------------------
class AttnPool(nn.Module):
    """Learned attention pooling over the sequence (mask-aware)."""

    def __init__(self, d_model: int):
        super().__init__()
        self.score = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # x: (B, L, D); mask: (B, L) with 1 for real tokens
        s = self.score(x).squeeze(-1)
        s = s.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(s, dim=1).unsqueeze(-1)
        return (x * w).sum(1)


class BiMambaHybridClassifier(nn.Module):
    """
    PonziMamba v2:
      - forward AND backward Mamba stacks (bidirectional context)
      - attention pooling instead of mean pooling
      - optional lexical branch fusing a token-frequency vector
        (the classical-baseline signal) into the classifier head
    All components are switchable for ablations.
    """

    def __init__(self, vocab_size: int, d_model: int = 128, n_layers: int = 2,
                 d_state: int = 16, dropout: float = 0.2, pad_id: int = 0,
                 lex_dim: int = 0, bidir: bool = True, attn_pool: bool = True):
        super().__init__()
        self.pad_id = pad_id
        self.bidir = bidir
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)

        def stack():
            return (nn.ModuleList([MambaBlock(d_model, d_state)
                                   for _ in range(n_layers)]),
                    nn.ModuleList([nn.LayerNorm(d_model)
                                   for _ in range(n_layers)]))

        self.blocks_f, self.norms_f = stack()
        if bidir:
            self.blocks_b, self.norms_b = stack()
        self.final_norm = nn.LayerNorm(d_model * (2 if bidir else 1))
        self.pool_f = AttnPool(d_model) if attn_pool else None
        self.pool_b = (AttnPool(d_model) if (attn_pool and bidir) else None)
        self.attn_pool = attn_pool

        feat_dim = d_model * (2 if bidir else 1)
        self.lex_dim = lex_dim
        if lex_dim > 0:
            self.lex_net = nn.Sequential(
                nn.Linear(lex_dim, d_model), nn.ReLU(), nn.Dropout(dropout))
            feat_dim += d_model
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(feat_dim, 1)

    def _run(self, x, blocks, norms):
        for norm, block in zip(norms, blocks):
            x = x + block(norm(x))
        return x

    def _pool(self, x, mask, pool):
        if self.attn_pool:
            return pool(x, mask)
        m = mask.float().unsqueeze(-1)
        return (x * m).sum(1) / m.sum(1).clamp(min=1.0)

    def forward(self, ids: torch.Tensor,
                lex: torch.Tensor = None) -> torch.Tensor:
        mask = (ids != self.pad_id)
        x = self.embed(ids)
        h_f = self._run(x, self.blocks_f, self.norms_f)
        feats = [self._pool(h_f, mask, self.pool_f)]
        if self.bidir:
            ids_rev = torch.flip(ids, dims=[1])
            x_b = self.embed(ids_rev)
            h_b = self._run(x_b, self.blocks_b, self.norms_b)
            feats.append(self._pool(h_b, torch.flip(mask, dims=[1]),
                                    self.pool_b))
        z = self.final_norm(torch.cat(feats, dim=-1))
        if self.lex_dim > 0:
            if lex is None:
                raise ValueError("lex features required (lex_dim > 0)")
            z = torch.cat([z, self.lex_net(lex)], dim=-1)
        return self.head(self.dropout(z)).squeeze(-1)


## 3 · Training & evaluation utilities
Solidity tokenizer, stratified splits / k-fold CV, metrics (Ponzi-class P/R/F1, ROC-AUC,
PR-AUC, confusion matrix), decision-threshold selection **on validation only**,
XGBoost + deep-model training loops with early stopping and efficiency logging.

In [6]:
import argparse
import json
import os
import platform
import random
import re
import time
from collections import Counter

import numpy as np

DATA_PATH = os.path.join("data", "ponzi_clean.jsonl")
RESULTS_PATH = "results.jsonl"

TOKEN_RE = re.compile(
    r"//[^\n]*|/\*.*?\*/"
    r"|\"(?:\\.|[^\"\\])*\"|'(?:\\.|[^'\\])*'"
    r"|0x[0-9a-fA-F]+|\d+\.?\d*"
    r"|[A-Za-z_$][A-Za-z0-9_$]*"
    r"|[^\sA-Za-z0-9_$]",
    re.DOTALL,
)


def tokenize(src: str):
    out = []
    for m in TOKEN_RE.finditer(src):
        t = m.group(0)
        if t.startswith("//") or t.startswith("/*"):
            continue
        if t[0] in "\"'":
            out.append("<STR>")
        elif t[0].isdigit() or t.startswith("0x"):
            out.append("<NUM>")
        else:
            out.append(t)
    return out


SOLIDITY_KEYWORDS = frozenset("""
pragma solidity contract library interface function modifier event struct enum
mapping address bool string bytes byte int uint
int8 int16 int32 int64 int128 int256 uint8 uint16 uint32 uint64 uint128 uint256
bytes1 bytes2 bytes4 bytes8 bytes16 bytes32
public private internal external pure view payable constant anonymous indexed
memory storage calldata returns return if else for while do break continue
new delete emit require assert revert throw this super selfdestruct suicide
msg sender value data gas tx origin gasprice block coinbase difficulty gaslimit
number timestamp now abi encode encodePacked decode keccak256 sha256 sha3
ripemd160 ecrecover addmod mulmod balance transfer send call delegatecall
callcode staticcall push pop length true false wei szabo finney ether
seconds minutes hours days weeks years var is using assembly let
constructor fallback receive virtual override abstract immutable unchecked
type error try catch <NUM> <STR> <PAD> <UNK>
""".split())


def canonicalize(tokens, rng=None, max_ids=256):
    """Alpha-rename user identifiers to ID0..ID{max_ids-1} by first
    occurrence. If rng is given, the ID assignment is randomly permuted
    (renaming augmentation): the code structure is unchanged but the
    model cannot memorize specific identifier names."""
    mapping, out = {}, []
    for t in tokens:
        if (t[0].isalpha() or t[0] in "_$") and t not in SOLIDITY_KEYWORDS:
            if t not in mapping:
                mapping[t] = len(mapping)
            out.append(mapping[t])
        else:
            out.append(t)
    k = len(mapping)
    order = list(range(k))
    if rng is not None:
        rng.shuffle(order)
    return [f"ID{order[t] % max_ids}" if isinstance(t, int) else t
            for t in out]


_TOK_CACHE = {}


def tokenize_rec(r, cap=None):
    """Tokenize a record once per session (cached by contract idx)."""
    idx = r.get("idx")
    toks = _TOK_CACHE.get(idx)
    if toks is None:
        toks = tokenize(r["sourcecode"])
        if idx is not None:
            _TOK_CACHE[idx] = toks
    return toks[:cap] if cap is not None else toks


def already_done(cfg):
    """Return the stored summary if an identical experiment config is
    already in results.jsonl (makes 'Run All' safe and idempotent)."""
    if not os.path.exists(RESULTS_PATH):
        return None
    probe_cfg = {k: v for k, v in cfg.items() if k != "force"}
    for line in open(RESULTS_PATH):
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue
        stored = {k: v for k, v in r.get("config", {}).items()
                  if k != "force"}
        if stored == probe_cfg:
            return r["summary"]
    return None


def load_data():
    if not os.path.exists(DATA_PATH):
        raise SystemExit("data/ponzi_clean.jsonl not found - run "
                         "`python prepare_data.py` first.")
    records = [json.loads(line) for line in open(DATA_PATH)]
    print(f"Loaded {len(records)} contracts "
          f"({sum(r['label'] for r in records)} Ponzi)")
    return records


def report_truncation(records, max_len):
    lens = [len(tokenize_rec(r)) for r in records]
    lens = np.array(lens)
    frac = float((lens > max_len).mean())
    stats = {"median_tokens": int(np.median(lens)),
             "p90_tokens": int(np.percentile(lens, 90)),
             "max_tokens": int(lens.max()),
             "frac_truncated_at_max_len": round(frac, 4)}
    print(f"Token-length stats: {stats}")
    return stats


# --------------------------------------------------------------------------
# Splits
# --------------------------------------------------------------------------
def stratified_split(records, seed, frac=(0.7, 0.1, 0.2)):
    rng = random.Random(seed)
    pos = [r for r in records if r["label"] == 1]
    neg = [r for r in records if r["label"] == 0]
    rng.shuffle(pos)
    rng.shuffle(neg)

    def cut(lst):
        n = len(lst)
        a, b = int(frac[0] * n), int((frac[0] + frac[1]) * n)
        return lst[:a], lst[a:b], lst[b:]

    p_tr, p_va, p_te = cut(pos)
    n_tr, n_va, n_te = cut(neg)
    train, valid, test = p_tr + n_tr, p_va + n_va, p_te + n_te
    for split in (train, valid, test):
        rng.shuffle(split)
    return train, valid, test


def cv_folds(records, k, seed):
    """Yield (train, valid, test) per fold; valid is 10% carved from train."""
    from sklearn.model_selection import StratifiedKFold, train_test_split
    y = [r["label"] for r in records]
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    idx = np.arange(len(records))
    for tr_idx, te_idx in skf.split(idx, y):
        tr_recs = [records[i] for i in tr_idx]
        te_recs = [records[i] for i in te_idx]
        tr, va = train_test_split(
            tr_recs, test_size=0.1, random_state=seed,
            stratify=[r["label"] for r in tr_recs])
        yield tr, va, te_recs


# --------------------------------------------------------------------------
# Metrics & threshold selection
# --------------------------------------------------------------------------
def compute_metrics(y_true, y_prob, threshold):
    from sklearn.metrics import (average_precision_score, confusion_matrix,
                                 f1_score, precision_score, recall_score,
                                 roc_auc_score)
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "confusion": {"tn": int(tn), "fp": int(fp),
                      "fn": int(fn), "tp": int(tp)},
    }


def best_threshold(y_true, y_prob):
    from sklearn.metrics import precision_recall_curve
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
    i = int(np.nanargmax(f1[:-1])) if len(thr) else 0
    return float(thr[i]) if len(thr) else 0.5


def finalize(y_va, p_va, y_te, p_te):
    thr = best_threshold(y_va, p_va)
    m = compute_metrics(y_te, p_te, thr)
    m["threshold"] = round(thr, 4)
    m["at_0.5"] = {k: round(v, 4) for k, v in
                   compute_metrics(y_te, p_te, 0.5).items()
                   if isinstance(v, float)}
    m["support_pos"] = int(np.sum(y_te))
    m["support_total"] = int(len(y_te))
    m["preds"] = {"y": [int(v) for v in y_te],
                  "p": [round(float(v), 4) for v in p_te],
                  "y_val": [int(v) for v in y_va],
                  "p_val": [round(float(v), 4) for v in p_va]}
    return m


# --------------------------------------------------------------------------
# XGBoost baseline
# --------------------------------------------------------------------------
def run_xgboost(train, valid, test, args, seed):
    from sklearn.feature_extraction.text import TfidfVectorizer
    import xgboost as xgb

    def to_text(rs):
        return [" ".join(tokenize_rec(r, args.max_len)) for r in rs]

    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=20000,
                          token_pattern=r"\S+")
    X_tr = vec.fit_transform(to_text(train))
    X_va = vec.transform(to_text(valid))
    X_te = vec.transform(to_text(test))
    y_tr = [r["label"] for r in train]
    y_va = [r["label"] for r in valid]
    y_te = [r["label"] for r in test]

    spw = (len(y_tr) - sum(y_tr)) / max(sum(y_tr), 1)
    clf = xgb.XGBClassifier(
        n_estimators=600, max_depth=6, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9,
        scale_pos_weight=spw, eval_metric="logloss",
        early_stopping_rounds=40,
        random_state=seed, n_jobs=-1,
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    p_va = clf.predict_proba(X_va)[:, 1]
    p_te = clf.predict_proba(X_te)[:, 1]
    return finalize(y_va, p_va, y_te, p_te)


# --------------------------------------------------------------------------
# Torch models
# --------------------------------------------------------------------------
def run_torch(train, valid, test, args, seed):
    import torch
    from torch.utils.data import DataLoader, Dataset

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    device = ("cuda" if torch.cuda.is_available() else
              "mps" if getattr(torch.backends, "mps", None)
              and torch.backends.mps.is_available() else "cpu")
    print(f"  device: {device}")
    if device == "cpu":
        print("  " + "!" * 62)
        print("  ! WARNING: no GPU detected - deep models on CPU can take")
        print("  ! HOURS PER EPOCH at max_len>=512. Install the CUDA build")
        print("  ! of PyTorch (pytorch.org) or use Colab with a GPU runtime.")
        print("  ! For a quick CPU sanity check use: max_len=128, d_model=64,")
        print("  ! epochs=3. Interrupt the kernel now if this is unintended.")
        print("  " + "!" * 62)
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    counter = Counter()
    for r in train:
        counter.update(tokenize_rec(r, args.max_len))
    itos = ["<PAD>", "<UNK>"] + [t for t, c in counter.most_common(args.vocab)
                                 if c >= 2]
    stoi = {t: i for i, t in enumerate(itos)}
    print(f"  vocab size: {len(itos)}")

    def encode(r):
        ids = [stoi.get(t, 1) for t in tokenize_rec(r, args.max_len)]
        return ids + [0] * (args.max_len - len(ids))

    class DS(Dataset):
        def __init__(self, rs):
            self.x = torch.tensor([encode(r) for r in rs], dtype=torch.long)
            self.y = torch.tensor([r["label"] for r in rs], dtype=torch.float)

        def __len__(self):
            return len(self.y)

        def __getitem__(self, i):
            return self.x[i], self.y[i]

    dl_tr = DataLoader(DS(train), batch_size=args.batch_size, shuffle=True)
    dl_va = DataLoader(DS(valid), batch_size=args.batch_size)
    dl_te = DataLoader(DS(test), batch_size=args.batch_size)

    if args.model == "mamba":
        model = MambaClassifier(len(itos), d_model=args.d_model,
                                n_layers=args.n_layers)
    elif args.model == "transformer":
        model = TransformerClassifier(len(itos), d_model=args.d_model,
                                      n_layers=args.n_layers,
                                      max_len=args.max_len)
    else:
        model = TextCNNClassifier(len(itos), d_model=args.d_model)
    model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  parameters: {n_params/1e6:.2f}M")

    criterion = (FocalLoss(alpha=args.alpha, gamma=args.gamma)
                 if args.loss == "focal"
                 else torch.nn.BCEWithLogitsLoss())
    optim = torch.optim.AdamW(model.parameters(), lr=args.lr,
                              weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, args.epochs)

    def evaluate(dl):
        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for xb, yb in dl:
                logits = model(xb.to(device))
                ps.extend(torch.sigmoid(logits).cpu().tolist())
                ys.extend(yb.tolist())
        return ys, ps

    best_f1, best_state, patience = -1.0, None, 0
    tokens_seen, train_time = 0, 0.0
    n_batches = len(dl_tr)
    run_start = time.time()
    for epoch in range(args.epochs):
        model.train()
        t0, total = time.time(), 0.0
        for bi, (xb, yb) in enumerate(dl_tr):
            optim.zero_grad()
            loss = criterion(model(xb.to(device)), yb.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            total += loss.item() * len(yb)
            tokens_seen += xb.numel()
            if epoch == 0 and bi == 0:
                eta = (time.time() - t0) * n_batches
                print(f"    first batch done - est. {eta/60:.1f} min/epoch "
                      f"({n_batches} batches); worst case this fold "
                      f"~{eta*args.epochs/60:.0f} min")
            elif bi % 20 == 0:
                print(f"    batch {bi}/{n_batches} "
                      f"({(time.time()-t0)/60:.1f} min)", end="\r")
        if device == "cuda":
            torch.cuda.synchronize()
        train_time += time.time() - t0
        sched.step()
        ys, ps = evaluate(dl_va)
        thr = best_threshold(ys, ps)
        m = compute_metrics(ys, ps, thr)
        print(f"  epoch {epoch+1:02d} loss={total/len(train):.4f} "
              f"val_f1={m['f1']:.4f} val_auc={m['roc_auc']:.4f} "
              f"({time.time()-t0:.1f}s)")
        if m["f1"] > best_f1:
            best_f1, patience = m["f1"], 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= args.patience:
                print("  early stop")
                break
        if args.time_budget_min and (time.time() - run_start) > \
                args.time_budget_min * 60:
            print("  TIME BUDGET reached - stopping this fold with the "
                  "best model so far")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    y_va, p_va = evaluate(dl_va)
    y_te, p_te = evaluate(dl_te)
    metrics = finalize(y_va, p_va, y_te, p_te)
    metrics["n_params"] = n_params
    metrics["train_tokens_per_sec"] = int(tokens_seen / max(train_time, 1e-9))
    if device == "cuda":
        metrics["peak_gpu_mem_mb"] = round(
            torch.cuda.max_memory_allocated() / 2**20, 1)
        del model, best_state, optim
        torch.cuda.empty_cache()   # release memory before the next fold
    return metrics


# --------------------------------------------------------------------------
def run_torch_v2(train, valid, test, args, seed):
    """PonziMamba v2: bidirectional Mamba + attention pooling + lexical
    branch + identifier canonicalization/renaming augmentation + balanced
    sampling + optional per-fold seed ensembling. All parts switchable."""
    import torch
    from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    device = ("cuda" if torch.cuda.is_available() else
              "mps" if getattr(torch.backends, "mps", None)
              and torch.backends.mps.is_available() else "cpu")
    print(f"  device: {device}")
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    # raw token lists (truncated) once per record
    def raw(r):
        return list(tokenize_rec(r, args.max_len))
    tok_tr = [raw(r) for r in train]
    tok_va = [raw(r) for r in valid]
    tok_te = [raw(r) for r in test]

    # vocab over (canonicalized or raw) train tokens
    counter = Counter()
    for toks in tok_tr:
        counter.update(canonicalize(toks) if args.canon else toks)
    itos = ["<PAD>", "<UNK>"] + [t for t, c in counter.most_common(args.vocab)
                                 if c >= 2]
    stoi = {t: i for i, t in enumerate(itos)}
    print(f"  vocab size: {len(itos)} (canon={bool(args.canon)})")

    # lexical branch features from RAW tokens (keeps identifier names)
    lex_dim = 0
    if args.hybrid:
        from sklearn.feature_extraction.text import CountVectorizer
        vec = CountVectorizer(max_features=2000, token_pattern=r"\S+")
        Xtr = vec.fit_transform(" ".join(t) for t in tok_tr).toarray()
        Xtr = np.log1p(Xtr).astype(np.float32)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
        def lexfeat(tok_lists):
            X = vec.transform(" ".join(t) for t in tok_lists).toarray()
            return ((np.log1p(X) - mu) / sd).astype(np.float32)
        lex_tr, lex_va, lex_te = ((Xtr - mu) / sd).astype(np.float32), \
            lexfeat(tok_va), lexfeat(tok_te)
        lex_dim = lex_tr.shape[1]
        print(f"  lexical branch: {lex_dim} features")

    class DS(Dataset):
        def __init__(self, toks, recs, lex, train_mode):
            self.toks, self.lex, self.train_mode = toks, lex, train_mode
            self.y = torch.tensor([r["label"] for r in recs],
                                  dtype=torch.float)

        def __len__(self):
            return len(self.y)

        def encode(self, toks):
            ids = [stoi.get(t, 1) for t in toks]
            return ids + [0] * (args.max_len - len(ids))

        def __getitem__(self, i):
            toks = self.toks[i]
            if args.canon:
                rng = random.Random() if (self.train_mode and
                                          args.canon_aug) else None
                toks = canonicalize(toks, rng=rng)
            ids = torch.tensor(self.encode(toks), dtype=torch.long)
            lex = (torch.from_numpy(self.lex[i]) if lex_dim
                   else torch.zeros(0))
            return ids, lex, self.y[i]

    ds_tr = DS(tok_tr, train, lex_tr if lex_dim else None, True)
    ds_va = DS(tok_va, valid, lex_va if lex_dim else None, False)
    ds_te = DS(tok_te, test, lex_te if lex_dim else None, False)

    if args.balanced:
        y_tr = np.array([r["label"] for r in train])
        w = np.where(y_tr == 1, 1.0 / max(y_tr.sum(), 1),
                     1.0 / max((1 - y_tr).sum(), 1))
        sampler = WeightedRandomSampler(
            torch.as_tensor(w, dtype=torch.double), len(train),
            replacement=True)
        dl_tr = DataLoader(ds_tr, batch_size=args.batch_size,
                           sampler=sampler)
    else:
        dl_tr = DataLoader(ds_tr, batch_size=args.batch_size, shuffle=True)
    dl_va = DataLoader(ds_va, batch_size=args.batch_size)
    dl_te = DataLoader(ds_te, batch_size=args.batch_size)

    criterion = (FocalLoss(alpha=args.alpha, gamma=args.gamma)
                 if args.loss == "focal"
                 else torch.nn.BCEWithLogitsLoss())

    def evaluate(model, dl):
        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for xb, lb, yb in dl:
                logits = model(xb.to(device),
                               lb.to(device) if lex_dim else None)
                ps.extend(torch.sigmoid(logits).cpu().tolist())
                ys.extend(yb.tolist())
        return ys, np.array(ps)

    n_batches = len(dl_tr)
    ens_va, ens_te, n_params = [], [], 0
    tokens_seen, train_time = 0, 0.0
    run_start = time.time()
    budget_hit = False
    for mi in range(args.n_models):
        if budget_hit:
            break
        torch.manual_seed(seed * 1000 + mi)
        model = BiMambaHybridClassifier(
            len(itos), d_model=args.d_model, n_layers=args.n_layers,
            lex_dim=lex_dim, bidir=bool(args.bidir),
            attn_pool=bool(args.attn_pool)).to(device)
        n_params = sum(p.numel() for p in model.parameters())
        if mi == 0:
            print(f"  parameters: {n_params/1e6:.2f}M x {args.n_models} models")
        optim = torch.optim.AdamW(model.parameters(), lr=args.lr,
                                  weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, args.epochs)
        best_f1, best_state, patience = -1.0, None, 0
        for epoch in range(args.epochs):
            model.train()
            t0, total = time.time(), 0.0
            for bi, (xb, lb, yb) in enumerate(dl_tr):
                optim.zero_grad()
                logits = model(xb.to(device),
                               lb.to(device) if lex_dim else None)
                loss = criterion(logits, yb.to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim.step()
                total += loss.item() * len(yb)
                tokens_seen += xb.numel()
                if mi == 0 and epoch == 0 and bi == 0:
                    eta = (time.time() - t0) * n_batches
                    worst = eta * args.epochs * args.n_models / 60
                    print(f"    first batch done - est. {eta/60:.1f} "
                          f"min/epoch ({n_batches} batches); worst case "
                          f"this fold ~{worst:.0f} min "
                          f"({args.n_models} model(s))")
                elif bi % 20 == 0:
                    print(f"    batch {bi}/{n_batches}", end="\r")
            if device == "cuda":
                torch.cuda.synchronize()
            train_time += time.time() - t0
            sched.step()
            ys, ps = evaluate(model, dl_va)
            m = compute_metrics(ys, ps, best_threshold(ys, ps))
            print(f"  [m{mi}] epoch {epoch+1:02d} loss={total/len(train):.4f}"
                  f" val_f1={m['f1']:.4f} val_auc={m['roc_auc']:.4f}"
                  f" ({time.time()-t0:.1f}s)")
            if m["f1"] > best_f1:
                best_f1, patience = m["f1"], 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                patience += 1
                if patience >= args.patience:
                    print("  early stop")
                    break
            if args.time_budget_min and (time.time() - run_start) > \
                    args.time_budget_min * 60:
                print("  TIME BUDGET reached - stopping with best so far")
                budget_hit = True
                break
        if best_state is not None:
            model.load_state_dict(best_state)
        y_va, p_va = evaluate(model, dl_va)
        y_te, p_te = evaluate(model, dl_te)
        ens_va.append(p_va)
        ens_te.append(p_te)
        if device == "cuda":
            del model, best_state, optim
            torch.cuda.empty_cache()

    p_va = np.mean(ens_va, axis=0)
    p_te = np.mean(ens_te, axis=0)
    metrics = finalize(y_va, p_va, y_te, p_te)
    metrics["n_params"] = n_params * args.n_models
    metrics["train_tokens_per_sec"] = int(tokens_seen / max(train_time, 1e-9))
    if device == "cuda":
        import torch as _t
        metrics["peak_gpu_mem_mb"] = round(
            _t.cuda.max_memory_allocated() / 2**20, 1)
    return metrics


def run_transfer(train, valid, test, args, seed):
    """v3: transfer learning from a pretrained code model (default
    GraphCodeBERT), enhanced with hierarchical long-context encoding
    (n_chunks x chunk_len subtokens, attention-pooled), a lexical fusion
    branch, partial layer freezing, AMP, and optional seed ensembling.
    Set hf_model="__test__" for an offline tiny-random-model self-test."""
    import torch
    from torch.utils.data import DataLoader, Dataset

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    device = ("cuda" if torch.cuda.is_available() else
              "mps" if getattr(torch.backends, "mps", None)
              and torch.backends.mps.is_available() else "cpu")
    print(f"  device: {device}")
    if device == "cpu" and args.hf_model != "__test__":
        print("  !! WARNING: fine-tuning a 125M-param model on CPU is not "
              "realistic - use a GPU.")
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    K, CL = args.n_chunks, args.chunk_len

    if args.hf_model == "__test__":
        from transformers import RobertaConfig, RobertaModel

        class StubTok:
            cls_token_id, sep_token_id, pad_token_id = 0, 1, 2

            def __call__(self, text, add_special_tokens=False,
                         truncation=True, max_length=512):
                def one(t):
                    return [hash(w) % 197 + 3
                            for w in t.split()][:max_length]
                if isinstance(text, list):
                    return {"input_ids": [one(t) for t in text]}
                return {"input_ids": one(text)}

        tok = StubTok()
        cfg = RobertaConfig(
            vocab_size=200, hidden_size=32, num_hidden_layers=2,
            num_attention_heads=2, intermediate_size=64,
            max_position_embeddings=CL + 10)

        def make_encoder():
            torch.manual_seed(0)
            return RobertaModel(cfg)
    else:
        from transformers import AutoModel, AutoTokenizer
        print(f"  loading {args.hf_model} (downloads once, ~500 MB)...")
        tok = AutoTokenizer.from_pretrained(args.hf_model)

        def make_encoder():
            # fallback chain for repos without merged safetensors
            # (transformers requires torch>=2.6 to load .bin checkpoints)
            try:
                return AutoModel.from_pretrained(args.hf_model)
            except ValueError as e:
                if "torch" not in str(e) and "safetensors" not in str(e):
                    raise
                print("  .bin load blocked (torch<2.6); trying safetensors "
                      "conversion PR of the official repo...")
                try:
                    return AutoModel.from_pretrained(
                        args.hf_model, revision="refs/pr/3",
                        use_safetensors=True)
                except Exception:
                    pass
                if args.hf_model == "microsoft/graphcodebert-base":
                    print("  trying safetensors mirror "
                          "claudios/graphcodebert-base ...")
                    try:
                        return AutoModel.from_pretrained(
                            "claudios/graphcodebert-base",
                            use_safetensors=True)
                    except Exception:
                        pass
                raise SystemExit(
                    "Could not load weights with torch<2.6. Fix: upgrade "
                    "PyTorch, e.g.\n  pip install -U torch --index-url "
                    "https://download.pytorch.org/whl/cu126\nthen restart "
                    "the kernel.")

    CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id
    body = CL - 2

    def encode(src):
        ids = tok(src, add_special_tokens=False, truncation=True,
                  max_length=K * body)["input_ids"]
        chunk_ids, chunk_attn, chunk_mask = [], [], []
        for i in range(K):
            seg = ids[i * body:(i + 1) * body]
            arr = [CLS] + seg + [SEP]
            att = [1] * len(arr)
            arr += [PAD] * (CL - len(arr))
            att += [0] * (CL - len(att))
            chunk_ids.append(arr)
            chunk_attn.append(att)
            chunk_mask.append(1 if seg else 0)
        if not any(chunk_mask):
            chunk_mask[0] = 1
        return chunk_ids, chunk_attn, chunk_mask

    def raw_tokens(r):
        return tokenize_rec(r, 2048)

    lex_dim = 0
    if args.hybrid:
        from sklearn.feature_extraction.text import CountVectorizer
        tok_tr = [raw_tokens(r) for r in train]
        vec = CountVectorizer(max_features=2000, token_pattern=r"\S+")
        Xtr = np.log1p(vec.fit_transform(
            " ".join(t) for t in tok_tr).toarray()).astype(np.float32)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6

        def lexfeat(recs):
            X = vec.transform(
                " ".join(raw_tokens(r)) for r in recs).toarray()
            return ((np.log1p(X) - mu) / sd).astype(np.float32)

        lex_all = {"train": ((Xtr - mu) / sd).astype(np.float32),
                   "valid": lexfeat(valid), "test": lexfeat(test)}
        lex_dim = 2000
        print(f"  lexical branch: {lex_dim} features")

    if args.probe:
        return _probe_path(train, valid, test, args, seed, device,
                           tok, (CLS, SEP, PAD), make_encoder,
                           lex_all if lex_dim else None, lex_dim, K, CL)

    class DS(Dataset):
        def __init__(self, recs, split):
            enc = [encode(r["sourcecode"]) for r in recs]
            self.ids = torch.tensor([e[0] for e in enc], dtype=torch.long)
            self.attn = torch.tensor([e[1] for e in enc], dtype=torch.long)
            self.cmask = torch.tensor([e[2] for e in enc], dtype=torch.long)
            self.lex = (torch.from_numpy(lex_all[split]) if lex_dim
                        else torch.zeros(len(recs), 0))
            self.y = torch.tensor([r["label"] for r in recs],
                                  dtype=torch.float)

        def __len__(self):
            return len(self.y)

        def __getitem__(self, i):
            return (self.ids[i], self.attn[i], self.cmask[i],
                    self.lex[i], self.y[i])

    dl_tr = DataLoader(DS(train, "train"), batch_size=args.batch_size,
                       shuffle=True)
    dl_va = DataLoader(DS(valid, "valid"), batch_size=args.batch_size)
    dl_te = DataLoader(DS(test, "test"), batch_size=args.batch_size)

    class HierTransfer(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = make_encoder()
            H = self.enc.config.hidden_size
            # freeze embeddings + lower layers against overfitting
            if args.freeze_layers > 0:
                for p in self.enc.embeddings.parameters():
                    p.requires_grad_(False)
                for layer in self.enc.encoder.layer[:args.freeze_layers]:
                    for p in layer.parameters():
                        p.requires_grad_(False)
            self.pool = AttnPool(H)
            feat = H
            if lex_dim:
                self.lex_net = torch.nn.Sequential(
                    torch.nn.Linear(lex_dim, H), torch.nn.ReLU(),
                    torch.nn.Dropout(0.2))
                feat += H
            self.drop = torch.nn.Dropout(0.2)
            self.head = torch.nn.Linear(feat, 1)

        def forward(self, ids, attn, cmask, lex):
            B, K_, L_ = ids.shape
            out = self.enc(input_ids=ids.view(B * K_, L_),
                           attention_mask=attn.view(B * K_, L_))
            cls = out.last_hidden_state[:, 0].view(B, K_, -1)
            z = self.pool(cls, cmask)
            if lex_dim:
                z = torch.cat([z, self.lex_net(lex)], dim=-1)
            return self.head(self.drop(z)).squeeze(-1)

    criterion = (FocalLoss(alpha=args.alpha, gamma=args.gamma)
                 if args.loss == "focal"
                 else torch.nn.BCEWithLogitsLoss())
    use_amp = (device == "cuda")

    def evaluate(model, dl):
        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for ids, attn, cm, lx, yb in dl:
                with torch.autocast("cuda", torch.float16,
                                    enabled=use_amp):
                    logits = model(ids.to(device), attn.to(device),
                                   cm.to(device), lx.to(device))
                ps.extend(torch.sigmoid(logits.float()).cpu().tolist())
                ys.extend(yb.tolist())
        return ys, np.array(ps)

    n_batches = len(dl_tr)
    ens_va, ens_te, n_params = [], [], 0
    tokens_seen, train_time = 0, 0.0
    run_start = time.time()
    budget_hit = False
    for mi in range(args.n_models):
        if budget_hit:
            break
        torch.manual_seed(seed * 1000 + mi)
        model = HierTransfer().to(device)
        n_params = sum(p.numel() for p in model.parameters())
        n_train = sum(p.numel() for p in model.parameters()
                      if p.requires_grad)
        if mi == 0:
            print(f"  parameters: {n_params/1e6:.1f}M total, "
                  f"{n_train/1e6:.1f}M trainable x {args.n_models} model(s)")
        optim = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=args.lr, weight_decay=0.01)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, args.epochs)
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        best_f1, best_state, patience = -1.0, None, 0
        for epoch in range(args.epochs):
            model.train()
            t0, total = time.time(), 0.0
            for bi, (ids, attn, cm, lx, yb) in enumerate(dl_tr):
                optim.zero_grad()
                with torch.autocast("cuda", torch.float16,
                                    enabled=use_amp):
                    logits = model(ids.to(device), attn.to(device),
                                   cm.to(device), lx.to(device))
                    loss = criterion(logits.float(), yb.to(device))
                scaler.scale(loss).backward()
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optim)
                scaler.update()
                total += loss.item() * len(yb)
                tokens_seen += ids.numel()
                if mi == 0 and epoch == 0 and bi == 0:
                    eta = (time.time() - t0) * n_batches
                    worst = eta * args.epochs * args.n_models / 60
                    print(f"    first batch done - est. {eta/60:.1f} "
                          f"min/epoch ({n_batches} batches); worst case "
                          f"this fold ~{worst:.0f} min")
                elif bi % 20 == 0:
                    print(f"    batch {bi}/{n_batches}", end="\r")
            if device == "cuda":
                torch.cuda.synchronize()
            train_time += time.time() - t0
            sched.step()
            ys, ps = evaluate(model, dl_va)
            m = compute_metrics(ys, ps, best_threshold(ys, ps))
            print(f"  [m{mi}] epoch {epoch+1:02d} "
                  f"loss={total/len(train):.4f} val_f1={m['f1']:.4f} "
                  f"val_auc={m['roc_auc']:.4f} ({time.time()-t0:.1f}s)")
            if m["f1"] > best_f1:
                best_f1, patience = m["f1"], 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                patience += 1
                if patience >= args.patience:
                    print("  early stop")
                    break
            if args.time_budget_min and (time.time() - run_start) > \
                    args.time_budget_min * 60:
                print("  TIME BUDGET reached - stopping with best so far")
                budget_hit = True
                break
        if best_state is not None:
            model.load_state_dict(best_state)
        y_va, p_va = evaluate(model, dl_va)
        y_te, p_te = evaluate(model, dl_te)
        ens_va.append(p_va)
        ens_te.append(p_te)
        if device == "cuda":
            del model, best_state, optim
            torch.cuda.empty_cache()

    p_va = np.mean(ens_va, axis=0)
    p_te = np.mean(ens_te, axis=0)
    metrics = finalize(y_va, p_va, y_te, p_te)
    metrics["n_params"] = n_params * max(len(ens_te), 1)
    metrics["train_tokens_per_sec"] = int(tokens_seen / max(train_time, 1e-9))
    if device == "cuda":
        import torch as _t
        metrics["peak_gpu_mem_mb"] = round(
            _t.cuda.max_memory_allocated() / 2**20, 1)
    return metrics


_EMB_MEM = {}


def _probe_path(train, valid, test, args, seed, device, tok, special_ids,
                make_encoder, lex_all, lex_dim, K, CL):
    """Frozen-encoder probing: embed every contract ONCE with the pretrained
    encoder (disk-cached across folds, runs and ablations), then train only
    a small attention-pooling head (+ lexical branch). Seconds per fold."""
    import torch
    from torch.utils.data import DataLoader, TensorDataset

    CLS, SEP, PAD = special_ids
    tag = re.sub(r"[^A-Za-z0-9]+", "_", args.hf_model)
    cache_path = os.path.join("data", f"emb_{tag}_{K}x{CL}.npz")
    if cache_path in _EMB_MEM:                       # in-RAM across folds
        cache = _EMB_MEM[cache_path]
        print(f"  embedding cache: {len(cache)} contracts (in memory)")
    elif os.path.exists(cache_path):
        z = np.load(cache_path, allow_pickle=False)
        idxs, emb, cm = z["idx"], z["emb"], z["cmask"]  # decompress ONCE
        cache = {str(idxs[i]): (emb[i], cm[i]) for i in range(len(idxs))}
        _EMB_MEM[cache_path] = cache
        print(f"  embedding cache: {len(cache)} contracts loaded "
              f"({cache_path})")
    else:
        cache = {}
        _EMB_MEM[cache_path] = cache

    all_recs = {}
    for r in train + valid + test:
        all_recs[str(r["idx"])] = r
    missing = [r for idx, r in all_recs.items() if idx not in cache]
    if missing:
        print(f"  computing embeddings for {len(missing)} contracts "
              f"(one-time; cached to {cache_path})...")
        if device == "cpu" and args.hf_model != "__test__":
            print("  " + "!" * 66)
            print("  ! NO GPU DETECTED - this pass would take HOURS on CPU.")
            print("  ! If this machine has an NVIDIA GPU, your torch install")
            print("  ! is CPU-only. Fix (then restart the kernel):")
            print("  !   pip install -U torch --index-url "
                  "https://download.pytorch.org/whl/cu126")
            print("  " + "!" * 66)
        enc = make_encoder().to(device).eval()
        use_amp = device == "cuda"
        Hd = enc.config.hidden_size
        body_ = CL - 2

        # batched subtokenization, then a FLAT list of non-empty chunks only
        # (about a quarter of chunks are empty padding - skip them entirely)
        texts = [r["sourcecode"] for r in missing]
        all_ids = tok(texts, add_special_tokens=False, truncation=True,
                      max_length=K * body_)["input_ids"]
        embs = np.zeros((len(missing), K, Hd), dtype=np.float16)
        cmasks = np.zeros((len(missing), K), dtype=np.uint8)
        flat = []                       # (record_pos, chunk_k, ids)
        for i, ids in enumerate(all_ids):
            for k in range(K):
                seg = ids[k * body_:(k + 1) * body_]
                if seg or k == 0:
                    cmasks[i, k] = 1
                    flat.append((i, k, [CLS] + seg + [SEP]))
        print(f"    {len(flat)} non-empty chunks to encode "
              f"(skipped {len(missing)*K - len(flat)} empty)")

        t0, done, SB = time.time(), 0, (64 if device == "cuda" else 8)
        with torch.inference_mode():
            s = 0
            while s < len(flat):
                batch = flat[s:s + SB]
                L = max(len(b[2]) for b in batch)
                ids_t = torch.full((len(batch), L), PAD, dtype=torch.long)
                att_t = torch.zeros((len(batch), L), dtype=torch.long)
                for j, (_, _, arr) in enumerate(batch):
                    ids_t[j, :len(arr)] = torch.tensor(arr)
                    att_t[j, :len(arr)] = 1
                try:
                    with torch.autocast("cuda", torch.float16,
                                        enabled=use_amp):
                        out = enc(input_ids=ids_t.to(device),
                                  attention_mask=att_t.to(device))
                    cls = out.last_hidden_state[:, 0].float().cpu()
                except Exception as err:
                    if "out of memory" in str(err).lower() and SB > 4:
                        SB //= 2
                        if device == "cuda":
                            torch.cuda.empty_cache()
                        print(f"    (GPU OOM - reducing batch to {SB})")
                        continue
                    raise
                for j, (i, k, _) in enumerate(batch):
                    embs[i, k] = cls[j].numpy().astype(np.float16)
                done += len(batch)
                s += len(batch)
                el = time.time() - t0
                if done == len(batch) or (s // SB) % 10 == 0:
                    rate = done / el
                    print(f"    {done}/{len(flat)} chunks "
                          f"({rate:.0f}/s, ~{(len(flat)-done)/rate/60:.1f} "
                          f"min left)   ", end="\r")
        print()
        for i, r in enumerate(missing):
            cache[str(r["idx"])] = (embs[i], cmasks[i])
        del enc
        if device == "cuda":
            torch.cuda.empty_cache()
        keys = sorted(cache.keys())
        np.savez_compressed(
            cache_path, idx=np.array(keys),
            emb=np.stack([cache[k][0] for k in keys]),
            cmask=np.stack([cache[k][1] for k in keys]))
        print(f"  embeddings cached ({time.time()-t0:.0f}s)")

    H = next(iter(cache.values()))[0].shape[-1]

    def tensors(recs, split):
        emb = torch.tensor(np.stack(
            [cache[str(r["idx"])][0] for r in recs]), dtype=torch.float32)
        cm = torch.tensor(np.stack(
            [cache[str(r["idx"])][1] for r in recs]), dtype=torch.long)
        lex = (torch.from_numpy(lex_all[split]) if lex_dim
               else torch.zeros(len(recs), 0))
        y = torch.tensor([r["label"] for r in recs], dtype=torch.float)
        return TensorDataset(emb, cm, lex, y)

    dl_tr = DataLoader(tensors(train, "train"), batch_size=args.batch_size,
                       shuffle=True)
    dl_va = DataLoader(tensors(valid, "valid"), batch_size=256)
    dl_te = DataLoader(tensors(test, "test"), batch_size=256)

    class ProbeHead(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.pool = AttnPool(H)
            feat = H
            if lex_dim:
                self.lex_net = torch.nn.Sequential(
                    torch.nn.Linear(lex_dim, 128), torch.nn.ReLU(),
                    torch.nn.Dropout(0.3))
                feat += 128
            self.drop = torch.nn.Dropout(0.3)
            self.head = torch.nn.Linear(feat, 1)

        def forward(self, emb, cm, lex):
            z = self.pool(emb, cm)
            if lex_dim:
                z = torch.cat([z, self.lex_net(lex)], dim=-1)
            return self.head(self.drop(z)).squeeze(-1)

    criterion = (FocalLoss(alpha=args.alpha, gamma=args.gamma)
                 if args.loss == "focal"
                 else torch.nn.BCEWithLogitsLoss())

    def evaluate(model, dl):
        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for emb, cm, lx, yb in dl:
                logits = model(emb.to(device), cm.to(device), lx.to(device))
                ps.extend(torch.sigmoid(logits).cpu().tolist())
                ys.extend(yb.tolist())
        return ys, np.array(ps)

    ens_va, ens_te, n_params = [], [], 0
    t_start = time.time()
    for mi in range(args.n_models):
        torch.manual_seed(seed * 1000 + mi)
        model = ProbeHead().to(device)
        n_params = sum(p.numel() for p in model.parameters())
        optim = torch.optim.AdamW(model.parameters(), lr=args.lr,
                                  weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, args.epochs)
        best_f1, best_state, patience = -1.0, None, 0
        for epoch in range(args.epochs):
            model.train()
            for emb, cm, lx, yb in dl_tr:
                optim.zero_grad()
                loss = criterion(model(emb.to(device), cm.to(device),
                                       lx.to(device)), yb.to(device))
                loss.backward()
                optim.step()
            sched.step()
            ys, ps = evaluate(model, dl_va)
            m = compute_metrics(ys, ps, best_threshold(ys, ps))
            if m["f1"] > best_f1:
                best_f1, patience = m["f1"], 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                patience += 1
                if patience >= args.patience:
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        y_va, p_va = evaluate(model, dl_va)
        y_te, p_te = evaluate(model, dl_te)
        ens_va.append(p_va)
        ens_te.append(p_te)
    print(f"  probe heads trained: {args.n_models} model(s), "
          f"{n_params/1e3:.0f}K params each, {time.time()-t_start:.1f}s")

    p_va = np.mean(ens_va, axis=0)
    p_te = np.mean(ens_te, axis=0)
    metrics = finalize(y_va, p_va, y_te, p_te)
    metrics["n_params"] = n_params
    return metrics

In [7]:
def library_versions():
    v = {"python": platform.python_version()}
    try:
        import torch
        v["torch"] = torch.__version__
        v["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            v["gpu"] = torch.cuda.get_device_name(0)
    except Exception:
        pass
    try:
        import sklearn
        v["sklearn"] = sklearn.__version__
    except Exception:
        pass
    try:
        import xgboost
        v["xgboost"] = xgboost.__version__
    except Exception:
        pass
    return v

## 4 · Experiment runner
`run_experiment(model=..., cv=5, ...)` mirrors the CLI: runs all folds/seeds,
prints a summary, and appends one JSON line (config, versions, token stats,
per-fold metrics) to `results.jsonl`.

In [8]:
from types import SimpleNamespace

DEFAULTS = dict(model=None, loss="focal", alpha=0.75, gamma=2.0,
                max_len=512, vocab=10000, d_model=128, n_layers=2,
                epochs=30, patience=6, batch_size=32, lr=1e-3,
                cv=0, seeds="42", tag="",
                bidir=1, attn_pool=1, hybrid=1, canon=1,
                canon_aug=1, balanced=1, n_models=1, time_budget_min=0, force=0,
                hf_model="microsoft/graphcodebert-base", n_chunks=4,
                chunk_len=512, freeze_layers=6, probe=1)

def run_experiment(**overrides):
    cfg = dict(DEFAULTS)
    cfg.update(overrides)
    assert cfg["model"] in ("mamba", "mamba2", "transformer", "textcnn", "xgboost", "transfer")
    args = SimpleNamespace(**cfg)

    if not cfg.get("force"):
        prev = already_done(cfg)
        if prev is not None:
            print(f"ALREADY DONE ({cfg['model']}) - identical config in "
                  f"results.jsonl; skipping in 0s. Pass force=1 to re-run.")
            return prev

    records = load_data()
    len_stats = report_truncation(records, args.max_len)

    runs = []
    if args.cv > 0:
        base_seed = int(str(args.seeds).split(",")[0])
        for fold, (tr, va, te) in enumerate(cv_folds(records, args.cv, base_seed)):
            runs.append((f"fold{fold}", base_seed + fold, tr, va, te))
    else:
        for seed in [int(s) for s in str(args.seeds).split(",")]:
            tr, va, te = stratified_split(records, seed)
            runs.append((f"seed{seed}", seed, tr, va, te))

    all_metrics = []
    for name, seed, train, valid, test in runs:
        print(f"\n=== {args.model} | loss={args.loss} | max_len={args.max_len} | {name} ===")
        print(f"  split: {len(train)}/{len(valid)}/{len(test)} "
              f"(test Ponzi: {sum(r['label'] for r in test)})")
        t0 = time.time()
        try:
            runner = {"xgboost": run_xgboost, "mamba2": run_torch_v2, "transfer": run_transfer}.get(args.model, run_torch)
            m = runner(train, valid, test, args, seed)
        except Exception as err:
            if "out of memory" in str(err).lower():
                raise SystemExit(
                    "GPU OUT OF MEMORY. Fixes, in order: "
                    "(1) restart the kernel to clear leftover allocations, "
                    "(2) rerun with a smaller batch_size (16 or 8), "
                    "(3) reduce max_len. "
                    "Completed runs are already saved in results.jsonl.")
            raise
        m["runtime_sec"] = round(time.time() - t0, 1)
        m["run"] = name
        all_metrics.append(m)
        print("  TEST:", {k: round(v, 4) for k, v in m.items() if isinstance(v, float)})

    summary = {}
    for key in ["precision", "recall", "f1", "roc_auc", "pr_auc"]:
        vals = [m[key] for m in all_metrics]
        summary[key] = {"mean": round(float(np.mean(vals)), 4),
                        "std": round(float(np.std(vals)), 4)}

    record = {"config": cfg, "versions": library_versions(),
              "token_length_stats": len_stats, "per_run": all_metrics,
              "summary": summary,
              "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
    with open(RESULTS_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")
    print("\n================ SUMMARY ================")
    print(json.dumps(summary, indent=2))
    return summary

## 5 · Main comparison — 5-fold CV, max-len 1024
Median contract length is 867 tokens (p90 = 2,725), so 1024 covers most contracts.
Expect the XGBoost cell to take ~1–2 min; each deep model ~30–90 min on a laptop GPU
(early stopping usually ends runs well before 30 epochs). Lower `batch_size` to 16 if
GPU memory is tight.

In [9]:
run_experiment(model="xgboost", cv=5, max_len=1024)

ALREADY DONE (xgboost) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.8281, 'std': 0.1295},
 'recall': {'mean': 0.5891, 'std': 0.0968},
 'f1': {'mean': 0.6725, 'std': 0.0474},
 'roc_auc': {'mean': 0.9472, 'std': 0.0068},
 'pr_auc': {'mean': 0.7391, 'std': 0.0405}}

In [10]:
run_experiment(model="textcnn", cv=5, max_len=1024)

ALREADY DONE (textcnn) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.7765, 'std': 0.1442},
 'recall': {'mean': 0.6018, 'std': 0.0786},
 'f1': {'mean': 0.6617, 'std': 0.0515},
 'roc_auc': {'mean': 0.9481, 'std': 0.0112},
 'pr_auc': {'mean': 0.7251, 'std': 0.0432}}

In [11]:
run_experiment(model="transformer", cv=5, max_len=1024)

ALREADY DONE (transformer) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.7359, 'std': 0.1265},
 'recall': {'mean': 0.5, 'std': 0.072},
 'f1': {'mean': 0.5869, 'std': 0.0554},
 'roc_auc': {'mean': 0.9264, 'std': 0.0099},
 'pr_auc': {'mean': 0.6316, 'std': 0.0925}}

In [12]:
run_experiment(model="mamba", cv=5, max_len=1024)

ALREADY DONE (mamba) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.7291, 'std': 0.1152},
 'recall': {'mean': 0.5946, 'std': 0.1223},
 'f1': {'mean': 0.6378, 'std': 0.065},
 'roc_auc': {'mean': 0.945, 'std': 0.0083},
 'pr_auc': {'mean': 0.716, 'std': 0.0416}}

## 6 · Ablation 1 — focal loss vs plain BCE

In [13]:
run_experiment(model="mamba", cv=5, max_len=1024, loss="bce")

ALREADY DONE (mamba) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.7969, 'std': 0.0498},
 'recall': {'mean': 0.6208, 'std': 0.0843},
 'f1': {'mean': 0.6952, 'std': 0.0679},
 'roc_auc': {'mean': 0.9494, 'std': 0.0114},
 'pr_auc': {'mean': 0.7296, 'std': 0.0487}}

## 7 · Ablation 2 — sequence-length scaling (+ efficiency story)
The Transformer at 2048 gives the quadratic-vs-linear memory/speed contrast for the
efficiency claim. Use `batch_size=8` there if you hit out-of-memory.

In [14]:
run_experiment(model="mamba", cv=5, max_len=256)

ALREADY DONE (mamba) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.8097, 'std': 0.1369},
 'recall': {'mean': 0.5442, 'std': 0.1094},
 'f1': {'mean': 0.6329, 'std': 0.074},
 'roc_auc': {'mean': 0.904, 'std': 0.0195},
 'pr_auc': {'mean': 0.6562, 'std': 0.0744}}

In [15]:
run_experiment(model="mamba", cv=5, max_len=512)

ALREADY DONE (mamba) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.685, 'std': 0.1362},
 'recall': {'mean': 0.62, 'std': 0.1151},
 'f1': {'mean': 0.6286, 'std': 0.052},
 'roc_auc': {'mean': 0.9327, 'std': 0.0201},
 'pr_auc': {'mean': 0.6962, 'std': 0.0685}}

In [16]:
run_experiment(model="mamba", cv=5, max_len=2048, batch_size=16)

ALREADY DONE (mamba) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.7743, 'std': 0.11},
 'recall': {'mean': 0.5823, 'std': 0.0914},
 'f1': {'mean': 0.653, 'std': 0.052},
 'roc_auc': {'mean': 0.9615, 'std': 0.0078},
 'pr_auc': {'mean': 0.7468, 'std': 0.0449}}

In [17]:
run_experiment(model="transformer", cv=5, max_len=2048, batch_size=16)

ALREADY DONE (transformer) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.8546, 'std': 0.0852},
 'recall': {'mean': 0.5256, 'std': 0.0752},
 'f1': {'mean': 0.6495, 'std': 0.0786},
 'roc_auc': {'mean': 0.9497, 'std': 0.0103},
 'pr_auc': {'mean': 0.6852, 'std': 0.0803}}

## 9 · PonziMamba **v2** — the radical upgrade
Five changes over v1, each switchable for ablations:
**bidirectional** Mamba stacks (`bidir`), **attention pooling** (`attn_pool`),
**hybrid lexical branch** fusing token-frequency features into the head (`hybrid`),
**identifier canonicalization + renaming augmentation** (`canon`, `canon_aug`),
**balanced batch sampling** (`balanced`), plus optional per-fold **seed ensembling**
(`n_models=3` averages three models' probabilities).
BCE is the default loss here since the ablation showed focal loss hurt.

**Realistic cost on a laptop GPU:** main run ~3–5 h (2-model ensemble at max-len 512, hard-capped at 90 min/fold); each ablation ~1–2 h (capped at 45 min/fold). Every run prints a worst-case fold estimate after its first batch and a projected-remaining time after each fold — if the estimate looks wrong, interrupt immediately, nothing is lost.

Max-len 512 is used because your v1 results showed F1 at 512 ≈ 1024 while halving compute; an optional final 1024/3-model run is at the end of this section for overnight use.

In [18]:
# MAIN v2 run - realistic: ~3-5 h on a laptop GPU (budget-capped at 90 min/fold)
run_experiment(model="mamba2", cv=5, max_len=512, loss="bce", n_models=2,
               epochs=20, patience=4, time_budget_min=90)

ALREADY DONE (mamba2) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.8651, 'std': 0.0477},
 'recall': {'mean': 0.5635, 'std': 0.0925},
 'f1': {'mean': 0.6784, 'std': 0.0722},
 'roc_auc': {'mean': 0.9429, 'std': 0.0068},
 'pr_auc': {'mean': 0.7298, 'std': 0.0712}}

### v2 ablations — knock out one component at a time

In [19]:
run_experiment(model="mamba2", cv=5, max_len=512, loss="bce", bidir=0,
               epochs=15, patience=4, time_budget_min=45)

ALREADY DONE (mamba2) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.8627, 'std': 0.0739},
 'recall': {'mean': 0.576, 'std': 0.079},
 'f1': {'mean': 0.6876, 'std': 0.0686},
 'roc_auc': {'mean': 0.9472, 'std': 0.0036},
 'pr_auc': {'mean': 0.7067, 'std': 0.0663}}

In [20]:
run_experiment(model="mamba2", cv=5, max_len=512, loss="bce", attn_pool=0,
               epochs=15, patience=4, time_budget_min=45)

ALREADY DONE (mamba2) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.8566, 'std': 0.0677},
 'recall': {'mean': 0.5635, 'std': 0.108},
 'f1': {'mean': 0.6746, 'std': 0.0902},
 'roc_auc': {'mean': 0.9392, 'std': 0.0146},
 'pr_auc': {'mean': 0.7242, 'std': 0.0697}}

In [21]:
run_experiment(model="mamba2", cv=5, max_len=512, loss="bce", hybrid=0,
               epochs=15, patience=4, time_budget_min=45)

ALREADY DONE (mamba2) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.7423, 'std': 0.0931},
 'recall': {'mean': 0.5, 'std': 0.1212},
 'f1': {'mean': 0.5924, 'std': 0.1104},
 'roc_auc': {'mean': 0.8853, 'std': 0.0327},
 'pr_auc': {'mean': 0.5936, 'std': 0.0766}}

In [22]:
run_experiment(model="mamba2", cv=5, max_len=512, loss="bce", canon=0,
               epochs=15, patience=4, time_budget_min=45)

ALREADY DONE (mamba2) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.8394, 'std': 0.1452},
 'recall': {'mean': 0.5569, 'std': 0.0631},
 'f1': {'mean': 0.6609, 'std': 0.0568},
 'roc_auc': {'mean': 0.9343, 'std': 0.0069},
 'pr_auc': {'mean': 0.7267, 'std': 0.0782}}

In [23]:
run_experiment(model="mamba2", cv=5, max_len=512, loss="bce", balanced=0,
               epochs=15, patience=4, time_budget_min=45)

ALREADY DONE (mamba2) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.8406, 'std': 0.0668},
 'recall': {'mean': 0.5698, 'std': 0.089},
 'f1': {'mean': 0.6732, 'std': 0.0693},
 'roc_auc': {'mean': 0.9326, 'std': 0.015},
 'pr_auc': {'mean': 0.7287, 'std': 0.0555}}

In [24]:
# OPTIONAL overnight run: full-size final model (~8-12 h, capped 150 min/fold)
# run_experiment(model="mamba2", cv=5, max_len=1024, loss="bce", n_models=3,
#                epochs=20, patience=4, time_budget_min=150)

## 10 · Late fusion — ensemble any two runs
Averages saved per-fold probabilities of two runs (e.g., **v2 + XGBoost**), with the
fusion threshold chosen on validation predictions only. Requires runs made with this
notebook version (they store predictions); re-run the XGBoost cell if needed.
Both runs must use the same `cv` and be run in this same notebook session lineage
so folds match — the cell verifies label alignment before fusing.

In [25]:
run_experiment(model="xgboost", cv=5, max_len=1024, tag="for-fusion")

ALREADY DONE (xgboost) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.8281, 'std': 0.1295},
 'recall': {'mean': 0.5891, 'std': 0.0968},
 'f1': {'mean': 0.6725, 'std': 0.0474},
 'roc_auc': {'mean': 0.9472, 'std': 0.0068},
 'pr_auc': {'mean': 0.7391, 'std': 0.0405}}

In [26]:
import json as _json, numpy as _np

def fuse(model_a, model_b, tag_a="", tag_b="", w=0.5):
    runs = [_json.loads(l) for l in open(RESULTS_PATH)]
    def pick(name, tag):
        cand = [r for r in runs if r["config"]["model"] == name
                and (not tag or r["config"].get("tag") == tag)
                and "preds" in r["per_run"][0]]
        if not cand:
            raise SystemExit(f"No completed run with saved predictions for "
                             f"'{name}' yet - run its experiment cell first.")
        return cand[-1]
    A, B = pick(*model_a), pick(*model_b)
    fold_metrics = []
    for ra, rb in zip(A["per_run"], B["per_run"]):
        assert ra["preds"]["y"] == rb["preds"]["y"], "fold mismatch - rerun both in this notebook"
        pv = w * _np.array(ra["preds"]["p_val"]) + (1-w) * _np.array(rb["preds"]["p_val"])
        pt = w * _np.array(ra["preds"]["p"])     + (1-w) * _np.array(rb["preds"]["p"])
        thr = best_threshold(ra["preds"]["y_val"], pv)
        fold_metrics.append(compute_metrics(ra["preds"]["y"], pt, thr))
    print(f"FUSION {A['config']['model']} + {B['config']['model']} (w={w})")
    for k in ["precision", "recall", "f1", "roc_auc", "pr_auc"]:
        v = [m[k] for m in fold_metrics]
        print(f"  {k:<10} {_np.mean(v):.4f} ± {_np.std(v):.4f}")

fuse(("mamba2", ""), ("xgboost", "for-fusion"))

FUSION mamba2 + xgboost (w=0.5)
  precision  0.8275 ± 0.0948
  recall     0.6389 ± 0.0290
  f1         0.7175 ± 0.0313
  roc_auc    0.9501 ± 0.0050
  pr_auc     0.7674 ± 0.0527


## 11 · **v3 — Transfer learning**: Hierarchical GraphCodeBERT
The literature's transfer-learning SOTA (SourceP) fine-tunes GraphCodeBERT, but
RoBERTa-class encoders read only **512 subtokens** — most of a median contract
(~1,500+ subtokens) is never seen. v3 enhances the transfer recipe:
**hierarchical long-context encoding** (`n_chunks`×512 subtokens, attention-pooled),
the ablation-proven **lexical fusion branch**, threshold tuning, ensembling, and
XGBoost fusion.

**Fast by design (`probe=1`, default):** the pretrained encoder is *frozen* and every
contract is embedded exactly **once** — a one-time pass of ~10–15 min on a laptop GPU,
cached to `data/emb_*.npz` — after which each fold trains a tiny attention-pool +
lexical head in **seconds**, and ablations reuse the cache. Total for this whole
section: **~15–25 min**. Frozen features also overfit less on 158 positives.
Full fine-tuning (`probe=0`) remains available as the optional last cell
(realistically ~30+ min *per epoch* — overnight territory).

First run downloads the pretrained weights (~500 MB). If loading fails on
torch < 2.6, the code automatically falls back to safetensors sources and, if all
else fails, prints the one-line PyTorch upgrade command.

In [27]:
%pip install -q transformers

Note: you may need to restart the kernel to use updated packages.


In [28]:
# MAIN v3 run (probe mode): one-time embedding pass, then seconds per fold
run_experiment(model="transfer", cv=5, loss="bce", batch_size=64, lr=1e-3,
               epochs=40, patience=8, n_models=3, time_budget_min=30)

ALREADY DONE (transfer) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.7885, 'std': 0.0733},
 'recall': {'mean': 0.6325, 'std': 0.1299},
 'f1': {'mean': 0.6876, 'std': 0.0658},
 'roc_auc': {'mean': 0.9509, 'std': 0.0087},
 'pr_auc': {'mean': 0.7596, 'std': 0.0688}}

### v3 ablations — is the enhancement real?
`n_chunks=1` is exactly the SourceP-style truncated fine-tune (the literature baseline); `hybrid=0` removes the lexical branch.

In [29]:
# ablation: SourceP-style single-chunk (512-subtoken truncation) baseline
run_experiment(model="transfer", cv=5, loss="bce", batch_size=64, lr=1e-3,
               epochs=40, patience=8, n_models=3, n_chunks=1,
               time_budget_min=20)

ALREADY DONE (transfer) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.7597, 'std': 0.0668},
 'recall': {'mean': 0.6514, 'std': 0.1073},
 'f1': {'mean': 0.6923, 'std': 0.056},
 'roc_auc': {'mean': 0.9515, 'std': 0.007},
 'pr_auc': {'mean': 0.7583, 'std': 0.0656}}

In [30]:
# ablation: remove the lexical fusion branch (reuses the embedding cache)
run_experiment(model="transfer", cv=5, loss="bce", batch_size=64, lr=1e-3,
               epochs=40, patience=8, n_models=3, hybrid=0,
               time_budget_min=20)

ALREADY DONE (transfer) - identical config in results.jsonl; skipping in 0s. Pass force=1 to re-run.


{'precision': {'mean': 0.5116, 'std': 0.0553},
 'recall': {'mean': 0.5375, 'std': 0.1347},
 'f1': {'mean': 0.508, 'std': 0.0449},
 'roc_auc': {'mean': 0.8869, 'std': 0.0239},
 'pr_auc': {'mean': 0.5611, 'std': 0.0622}}

In [31]:
# OPTIONAL: full fine-tuning instead of probing (~30+ min/EPOCH on a laptop GPU)
# run_experiment(model="transfer", cv=5, loss="bce", probe=0, batch_size=4,
#                lr=2e-5, epochs=4, patience=2, time_budget_min=90)

### v3 + XGBoost late fusion — the expected overall winner

In [32]:
fuse(("transfer", ""), ("xgboost", "for-fusion"))

FUSION transfer + xgboost (w=0.5)
  precision  0.8175 ± 0.1320
  recall     0.6022 ± 0.1136
  f1         0.6762 ± 0.0622
  roc_auc    0.9371 ± 0.0143
  pr_auc     0.7374 ± 0.0584


## 8 · Results table
Summarizes every run recorded in `results.jsonl`. When all experiments are done,
send `results.jsonl` back for the full analysis and novelty assessment.

In [33]:
import os
if not os.path.exists(RESULTS_PATH):
    print("No results yet - run the experiment cells above first.")
else:
    import pandas as pd, json as _json
    rows = []
    for line in open(RESULTS_PATH):
        r = _json.loads(line)
        c, s = r["config"], r["summary"]
        eff = r["per_run"][0]
        rows.append({
            "model": c["model"], "loss": c["loss"], "max_len": c["max_len"],
            "protocol": f"cv{c['cv']}" if c.get("cv") else f"seeds {c['seeds']}",
            "F1": f"{s['f1']['mean']:.3f} ± {s['f1']['std']:.3f}",
            "Precision": f"{s['precision']['mean']:.3f}",
            "Recall": f"{s['recall']['mean']:.3f}",
            "ROC-AUC": f"{s['roc_auc']['mean']:.3f}",
            "PR-AUC": f"{s['pr_auc']['mean']:.3f}",
            "params(M)": round(eff.get("n_params", 0)/1e6, 2) or "-",
            "tok/s": eff.get("train_tokens_per_sec", "-"),
            "peakGPU(MB)": eff.get("peak_gpu_mem_mb", "-"),
        })
    pd.DataFrame(rows)

## 12 · Revision Part A — statistical analysis of saved predictions (CPU-only, ~2 min)

Added for the journal revision (Reviewer 1 points 2/6, Reviewer 2 points 1–3).
Reads `results.jsonl` (no re-training) and reproduces, deterministically (seed 42), every
number in **Table 2 (LNF rows), Table 3, and Table 5** of the revised manuscript:
per-fold F1 of the two LNF decision rules (fixed τ = 1/2, primary; validation-calibrated),
paired t / Wilcoxon tests, the paired contract-level bootstrap, every fusion pairing,
and the stacking meta-learner comparison.

In [34]:
# =====================================================================
# Part A - Statistical analysis of saved predictions (CPU-only, ~2 min)
# Reads results.jsonl written by the experiment cells above and computes
# every statistic reported in Section 6.2/6.5 and Table 3/Table 5 of the
# revised manuscript. Deterministic (seed 42): re-running reproduces the
# printed numbers exactly.
# =====================================================================
import json

try:
    import scipy  # noqa: F401
except ImportError:
    raise SystemExit("scipy missing - run:  %pip install -q scipy  "
                     "then re-run this cell")
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, f1_score,
                             precision_score, recall_score, roc_auc_score)

RESULTS_PATH = globals().get("RESULTS_PATH", "results.jsonl")
N_BOOT = 10_000
rng = np.random.default_rng(42)

runs = [json.loads(l) for l in open(RESULTS_PATH)]


def get(model, **kw):
    out = [r for r in runs if r["config"]["model"] == model
           and all(r["config"].get(k) == v for k, v in kw.items())]
    assert out, f"missing run: {model} {kw}"
    return out[-1]


MODELS = {
    "TF-IDF + XGBoost": get("xgboost", tag="for-fusion"),
    "TextCNN": get("textcnn", max_len=1024),
    "Transformer (L=1,024)": get("transformer", max_len=1024),
    "Transformer (L=2,048)": get("transformer", max_len=2048),
    "PonziMamba SSM": get("mamba", loss="bce", max_len=1024),
    "Enhanced SSM": [r for r in runs if r["config"]["model"] == "mamba2"
                     and r["config"].get("hybrid") == 1
                     and r["config"].get("n_models") == 2][-1],
    "Frozen probe (K=4)": get("transfer", n_chunks=4, hybrid=1),
}
XGB, PRB, SSM = (MODELS["TF-IDF + XGBoost"], MODELS["Frozen probe (K=4)"],
                 MODELS["PonziMamba SSM"])


def arr(r, key):
    return [np.array(m["preds"][key]) for m in r["per_run"]]


def best_threshold(y, p):
    cand = np.unique(p)
    f1s = [f1_score(y, (p >= t).astype(int), zero_division=0) for t in cand]
    return cand[int(np.argmax(f1s))]


def fuse(models, rule):
    """Average member probabilities; decide at tau=0.5 ('fixed') or at a
    threshold tuned only on the fused VALIDATION predictions ('valtuned')."""
    Y, Yv = arr(models[0], "y"), arr(models[0], "y_val")
    pts = [arr(m, "p") for m in models]
    pvs = [arr(m, "p_val") for m in models]
    recs = []
    for k in range(len(Y)):
        pt = np.mean([p[k] for p in pts], axis=0)
        pv = np.mean([p[k] for p in pvs], axis=0)
        t = 0.5 if rule == "fixed" else best_threshold(Yv[k], pv)
        yh = (pt >= t).astype(int)
        recs.append(dict(
            y=Y[k], p=pt, yh=yh, tau=t,
            f1=f1_score(Y[k], yh), prec=precision_score(Y[k], yh),
            rec=recall_score(Y[k], yh), auc=roc_auc_score(Y[k], pt),
            ap=average_precision_score(Y[k], pt)))
    return recs


def summ(recs):
    return {k: (float(np.mean([r[k] for r in recs])),
                float(np.std([r[k] for r in recs])))
            for k in ("prec", "rec", "f1", "auc", "ap")}


def fmt(recs):
    s = summ(recs)
    return ("P {0:.3f}±{1:.3f}  R {2:.3f}±{3:.3f}  F1 {4:.3f}±{5:.3f}  "
            "AUC {6:.3f}  PR-AUC {7:.3f}".format(
                *s["prec"], *s["rec"], *s["f1"], s["auc"][0], s["ap"][0]))


def boot_diff(pairs):
    """Paired contract-level bootstrap of the fold-mean F1 difference.
    pairs: per fold (y, yhat_A, yhat_B). Returns mean, lo, hi, one-sided p."""
    fa = np.zeros((N_BOOT, len(pairs)))
    fb = np.zeros((N_BOOT, len(pairs)))
    for k, (y, ya, yb) in enumerate(pairs):
        idx = rng.integers(0, len(y), size=(N_BOOT, len(y)))
        for out, yh in ((fa, ya), (fb, yb)):
            tp = (y * yh)[idx].sum(1)
            fp = ((1 - y) * yh)[idx].sum(1)
            fn = (y * (1 - yh))[idx].sum(1)
            out[:, k] = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    d = fa.mean(1) - fb.mean(1)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi), float((d <= 0).mean())


def model_decisions(r):
    return [(np.array(m["preds"]["y"]),
             (np.array(m["preds"]["p"]) >= m["threshold"]).astype(int))
            for m in r["per_run"]]


# ---------- 1. LNF under both decision rules ----------
LNF_FIX = fuse([PRB, XGB], "fixed")
LNF_CAL = fuse([PRB, XGB], "valtuned")
print("LNF (fixed tau=0.5):          ", fmt(LNF_FIX))
print("  per-fold F1:", [round(r["f1"], 3) for r in LNF_FIX])
print("LNF (validation-calibrated):  ", fmt(LNF_CAL))
print("  per-fold F1:", [round(r["f1"], 3) for r in LNF_CAL],
      " per-fold tau*:", [round(r["tau"], 3) for r in LNF_CAL])

# ---------- 2. per-fold F1 of every individual model ----------
print("\nPer-fold F1 (validation-tuned thresholds, as stored):")
for nm, r in MODELS.items():
    ff = [m["f1"] for m in r["per_run"]]
    print(f"  {nm:22s} {[round(v,3) for v in ff]}  "
          f"mean {np.mean(ff):.3f} ± {np.std(ff):.3f}")

# ---------- 3. paired fold-level tests + contract-level bootstrap ----------
lnf_f1 = np.array([r["f1"] for r in LNF_FIX])
lnf_yh = [(r["y"], r["yh"]) for r in LNF_FIX]
print("\nLNF (tau=0.5) vs each individual model:")
print(f"{'model':22s} {'dF1':>6s} {'t-p':>7s} {'Wlcx-p':>7s} {'wins':>5s} "
      f"{'boot dF1 [95% CI]':>24s} {'boot-p':>7s}")
for nm, r in MODELS.items():
    b = np.array([m["f1"] for m in r["per_run"]])
    d = lnf_f1 - b
    tt = stats.ttest_rel(lnf_f1, b)
    wz = stats.wilcoxon(lnf_f1, b, alternative="greater")
    mod = model_decisions(r)
    bm, lo, hi, bp = boot_diff([(a[0], a[1], c[1])
                                for a, c in zip(lnf_yh, mod)])
    print(f"{nm:22s} {d.mean():+.3f} {tt.pvalue:7.4f} {wz.pvalue:7.4f} "
          f"{int((d>0).sum())}/5  {bm:+.3f} [{lo:+.3f}, {hi:+.3f}] {bp:7.4f}")

# ---------- 4. fusion combinations (fixed tau=0.5) ----------
print("\nFusion combinations (probability averaging, fixed tau=0.5):")
for nm, ms in [("probe+XGB (LNF)", [PRB, XGB]), ("XGB+SSM", [XGB, SSM]),
               ("probe+SSM", [PRB, SSM]), ("3-way", [PRB, XGB, SSM])]:
    R = fuse(ms, "fixed")
    print(f"  {nm:16s} {[round(r['f1'],3) for r in R]}  "
          f"mean {np.mean([r['f1'] for r in R]):.3f} "
          f"± {np.std([r['f1'] for r in R]):.3f}")
print("probe+SSM (per-pair validation-calibrated), for transparency:")
R = fuse([PRB, SSM], "valtuned")
print(f"  {[round(r['f1'],3) for r in R]}  mean "
      f"{np.mean([r['f1'] for r in R]):.3f} ± {np.std([r['f1'] for r in R]):.3f}")
bm, lo, hi, bp = boot_diff([(a[0], a[1], c["yh"])
                            for a, c in zip(lnf_yh, R)])
print(f"  LNF(0.5) vs probe+SSM(cal): dF1={bm:+.3f} [{lo:+.3f},{hi:+.3f}] "
      f"p(one-sided)={bp:.4f}")

# ---------- 5. stacking meta-learner (alternative fusion, R2) ----------
Yv = arr(PRB, "y_val")
Xv = list(zip(arr(PRB, "p_val"), arr(XGB, "p_val")))
Xt = list(zip(arr(PRB, "p"), arr(XGB, "p")))
Y = arr(PRB, "y")
stk = []
for k in range(5):
    clf = LogisticRegression(max_iter=1000).fit(
        np.column_stack(Xv[k]), Yv[k])
    pt = clf.predict_proba(np.column_stack(Xt[k]))[:, 1]
    pv = clf.predict_proba(np.column_stack(Xv[k]))[:, 1]
    t = best_threshold(Yv[k], pv)
    yh = (pt >= t).astype(int)
    stk.append(dict(y=Y[k], yh=yh, f1=f1_score(Y[k], yh),
                    prec=precision_score(Y[k], yh),
                    rec=recall_score(Y[k], yh)))
print("\nStacking (logistic meta-learner fit on validation probabilities):")
print(f"  per-fold F1 {[round(r['f1'],3) for r in stk]}  "
      f"mean {np.mean([r['f1'] for r in stk]):.3f} "
      f"± {np.std([r['f1'] for r in stk]):.3f}")
bm, lo, hi, bp = boot_diff([(a[0], a[1], c["yh"])
                            for a, c in zip(lnf_yh, stk)])
print(f"  LNF(0.5) vs stacking: dF1={bm:+.3f} [{lo:+.3f}, {hi:+.3f}] "
      f"p(one-sided)={bp:.4f}")

# ---------- 6. deployment numbers (pooled, fixed tau=0.5) ----------
Yp = np.concatenate([r["y"] for r in LNF_FIX])
Hp = np.concatenate([r["yh"] for r in LNF_FIX])
print(f"\nPooled LNF (tau=0.5): precision {precision_score(Yp,Hp):.3f}  "
      f"recall {recall_score(Yp,Hp):.3f}  F1 {f1_score(Yp,Hp):.3f}  "
      f"alerts {int(Hp.sum())}  false alerts {int(((1-Yp)*Hp).sum())}")


LNF (fixed tau=0.5):           P 0.880±0.038  R 0.639±0.062  F1 0.738±0.044  AUC 0.956  PR-AUC 0.783
  per-fold F1: [0.807, 0.755, 0.68, 0.746, 0.704]
LNF (validation-calibrated):   P 0.865±0.042  R 0.645±0.112  F1 0.730±0.070  AUC 0.956  PR-AUC 0.783
  per-fold F1: [0.787, 0.807, 0.609, 0.746, 0.704]  per-fold tau*: [0.371, 0.321, 0.715, 0.536, 0.456]

Per-fold F1 (validation-tuned thresholds, as stored):
  TF-IDF + XGBoost       [0.609, 0.755, 0.679, 0.667, 0.654]  mean 0.672 ± 0.047
  TextCNN                [0.702, 0.706, 0.587, 0.702, 0.612]  mean 0.662 ± 0.051
  Transformer (L=1,024)  [0.667, 0.638, 0.561, 0.526, 0.542]  mean 0.587 ± 0.055
  Transformer (L=2,048)  [0.68, 0.745, 0.588, 0.704, 0.531]  mean 0.650 ± 0.079
  PonziMamba SSM         [0.746, 0.786, 0.667, 0.69, 0.588]  mean 0.695 ± 0.068
  Enhanced SSM           [0.741, 0.786, 0.596, 0.63, 0.64]  mean 0.678 ± 0.072
  Frozen probe (K=4)     [0.697, 0.759, 0.625, 0.758, 0.6]  mean 0.688 ± 0.066

LNF (tau=0.5) vs each indivi

## 13 · Revision Part B — benchmark variants: sensitivity & leakage (GPU)

Added for the journal revision (Reviewer 1 points 1/3/4, Reviewer 2 point 6).
Run the helper cell below once, then B1–B3 in order (each restores the strict
benchmark when it finishes, even on error). **Restart the kernel first** so no stale
GPU allocations remain, and re-run the Section 0–4 definition cells.

| Block | Produces | Approx. runtime (RTX 4060 Laptop) |
|---|---|---|
| B1 | Table 6 (Jaccard 0.80/0.85/0.90/0.95) | 30–45 min |
| B2 | Table 7 (conflict-resolution rules) | 20–30 min |
| B3 | Table 8 (conventional vs. strict, per model) | 2–4 h |

The embedding cache is keyed by contract id, so variants reuse previously computed
embeddings and only embed contracts not seen before.

In [35]:
# --- one-time patch: make JSON saving NumPy-safe (fixes
# "Object of type float32 is not JSON serializable" when run_experiment
# writes results.jsonl; some library versions return np.float32 values).
# Run AFTER the Section 0-4 definition cells, BEFORE Part B.
import json as _json_patch_mod
import numpy as _np_patch

if not getattr(_json_patch_mod, "_np_safe_patched", False):
    _orig_dumps = _json_patch_mod.dumps

    def _np_default(o):
        if isinstance(o, _np_patch.floating):
            return float(o)
        if isinstance(o, _np_patch.integer):
            return int(o)
        if isinstance(o, _np_patch.bool_):
            return bool(o)
        if isinstance(o, _np_patch.ndarray):
            return o.tolist()
        raise TypeError(f"Object of type {type(o).__name__} "
                        "is not JSON serializable")

    def _np_safe_dumps(obj, *a, **kw):
        kw.setdefault("default", _np_default)
        return _orig_dumps(obj, *a, **kw)

    _json_patch_mod.dumps = _np_safe_dumps
    _json_patch_mod._np_safe_patched = True
    print("json.dumps patched: NumPy scalars/arrays now serialize safely")
else:
    print("json.dumps already patched")

# =====================================================================
# Part B - Benchmark variants: audit sensitivity, conflict-rule
# sensitivity, and per-model leakage quantification (Revision, R1.1,
# R1.3, R1.4, R2.6). Requires the data-prep repo download from Section 1
# and the training utilities from Sections 2-4 to be defined (run those
# cells first in this kernel).
#
# What it does:
#   B1  rebuilds the benchmark at Jaccard 0.80/0.85/0.90/0.95  -> Table 6
#   B2  rebuilds it under 4 conflict-resolution rules           -> Table 7
#   B3  re-runs every model family on the conventional
#       (exact-dedup-only) benchmark vs. the strict one         -> Table 8
# Each block prints manuscript-ready rows. The canonical strict dataset
# is backed up first and restored afterwards, so the rest of the
# notebook is unaffected.
# =====================================================================
import json as _j
import os
import random as _random
import shutil
from collections import defaultdict

import numpy as _np

STRICT_BACKUP = os.path.join("data", "ponzi_clean_strict_backup.jsonl")


def _load_exact_dedup_records():
    """Replicates the exact-duplicate removal of prepare_data (Section 1)
    and returns the 5,917-contract pool from which every variant is
    built."""
    import hashlib
    import re as _re
    ds_dir = os.path.join(REPO_DIR, "dataset")
    labels = {}
    for fn in LABEL_FILES:
        with open(os.path.join(ds_dir, fn)) as f:
            for line in f:
                parts = line.split()
                if len(parts) == 2:
                    labels[parts[0]] = int(parts[1])
    seen, records = set(), []
    with open(os.path.join(ds_dir, "data.jsonl")) as fin:
        for line in fin:
            d = _j.loads(line)
            idx = str(d["idx"])
            if idx not in labels:
                continue
            src = d["sourcecode"]
            dig = hashlib.md5(
                _re.sub(r"\s+", "", src).encode()).hexdigest()
            if dig in seen:
                continue
            seen.add(dig)
            records.append({"idx": idx, "address": d.get("address", ""),
                            "label": labels[idx], "sourcecode": src})
    return records


_POOL = None
_PAIR_CACHE = {}


def build_benchmark(jaccard_thr=0.90, rule="keep_ponzi", seed=0,
                    write=True):
    """Rebuild the benchmark at a given near-duplicate threshold and
    conflict-resolution rule; optionally install it at DATA_PATH.
    rule in {keep_ponzi, drop_conflict, majority, random,
             keep_near_dups}. Returns an audit-stats dict."""
    global _POOL
    if _POOL is None:
        _POOL = _load_exact_dedup_records()
    records = list(_POOL)

    if rule == "keep_near_dups":              # conventional protocol
        kept = records
        stats = dict(thr=None, pairs=None, conflicts=None)
    else:
        if jaccard_thr not in _PAIR_CACHE:
            print(f"scanning near-duplicates at J >= {jaccard_thr} ...")
            _PAIR_CACHE[jaccard_thr] = find_near_duplicates(
                records, jaccard_thr=jaccard_thr)
        pairs = _PAIR_CACHE[jaccard_thr]
        conflicts = sum(1 for i, j, _ in pairs
                        if records[i]["label"] != records[j]["label"])
        parent = list(range(len(records)))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for i, j, _ in pairs:
            parent[find(i)] = find(j)
        groups = defaultdict(list)
        for i in range(len(records)):
            groups[find(i)].append(i)

        rng = _random.Random(seed)
        keep = set()
        for members in groups.values():
            labels_m = {records[i]["label"] for i in members}
            if rule == "drop_conflict" and len(labels_m) > 1:
                continue                      # discard whole component
            if rule == "keep_ponzi":
                members = sorted(members, key=lambda i: (
                    -records[i]["label"], records[i]["idx"]))
                keep.add(members[0])
            elif rule == "majority":
                pos = sum(records[i]["label"] for i in members)
                maj = 1 if 2 * pos >= len(members) else 0
                cand = sorted((i for i in members
                               if records[i]["label"] == maj),
                              key=lambda i: records[i]["idx"])
                keep.add(cand[0])
            elif rule == "random":
                keep.add(rng.choice(sorted(members)))
            else:                             # drop_conflict, clean comp
                members = sorted(members, key=lambda i: (
                    -records[i]["label"], records[i]["idx"]))
                keep.add(members[0])
        kept = [r for i, r in enumerate(records) if i in keep]
        stats = dict(thr=jaccard_thr, pairs=len(pairs),
                     conflicts=conflicts)

    stats.update(rule=rule, contracts=len(kept),
                 ponzi=sum(r["label"] for r in kept))
    if write:
        if os.path.exists(DATA_PATH) and not os.path.exists(STRICT_BACKUP):
            shutil.copy(DATA_PATH, STRICT_BACKUP)
        with open(DATA_PATH, "w") as f:
            for r in kept:
                f.write(_j.dumps(r) + "\n")
    print("benchmark:", stats)
    return stats


def restore_strict():
    if os.path.exists(STRICT_BACKUP):
        shutil.copy(STRICT_BACKUP, DATA_PATH)
        print("strict benchmark restored at", DATA_PATH)


def fuse_fixed_from_tags(tag_probe, tag_xgb):
    """LNF under the calibration-free rule (tau = 1/2) from the two most
    recent tagged runs; returns (mean, sd, per-fold list)."""
    from sklearn.metrics import f1_score
    runs = [_j.loads(l) for l in open(RESULTS_PATH)]

    def pick(model, tag):
        cand = [r for r in runs if r["config"]["model"] == model
                and r["config"].get("tag") == tag]
        assert cand, f"no run {model}/{tag}"
        return cand[-1]

    A = pick("transfer", tag_probe)
    B = pick("xgboost", tag_xgb)
    f1s = []
    for ra, rb in zip(A["per_run"], B["per_run"]):
        assert ra["preds"]["y"] == rb["preds"]["y"], "fold mismatch"
        p = (_np.array(ra["preds"]["p"]) + _np.array(rb["preds"]["p"])) / 2
        f1s.append(f1_score(ra["preds"]["y"], (p >= 0.5).astype(int)))
    return float(_np.mean(f1s)), float(_np.std(f1s)), [round(v, 3)
                                                       for v in f1s]


json.dumps patched: NumPy scalars/arrays now serialize safely


In [36]:
# --------------------------------------------------------------- B1
# Jaccard-threshold sensitivity  ->  manuscript Table 6
# Runtime: ~30-45 min total on an RTX 4060 Laptop (the XGBoost runs are
# seconds; each transfer run is one ~80 s embedding pass over the new
# corpus variant plus seconds per fold; embeddings are cached by
# contract id, so later variants reuse most of the cache).
rows = []
try:
    for thr in (0.80, 0.85, 0.90, 0.95):
        st = build_benchmark(jaccard_thr=thr, rule="keep_ponzi")
        tagx, tagp = f"sens-jac{int(thr*100)}-xgb", f"sens-jac{int(thr*100)}-probe"
        sx = run_experiment(model="xgboost", cv=5, max_len=1024, tag=tagx)
        sp = run_experiment(model="transfer", cv=5, loss="bce",
                            batch_size=64, lr=1e-3, epochs=40, patience=8,
                            n_models=3, time_budget_min=30, tag=tagp)
        m, s, ff = fuse_fixed_from_tags(tagp, tagx)
        rows.append((thr, st["pairs"], st["conflicts"], st["contracts"],
                     st["ponzi"],
                     f"{sx['f1']['mean']:.3f} ± {sx['f1']['std']:.3f}",
                     f"{m:.3f} ± {s:.3f}"))
finally:
    restore_strict()
print("\n=== Table 6 rows (Jaccard thr | pairs | conflicting | contracts"
      " | Ponzi | XGBoost F1 | LNF F1) ===")
for r in rows:
    print(" | ".join(str(x) for x in r))



scanning near-duplicates at J >= 0.8 ...
benchmark: {'thr': 0.8, 'pairs': 25064, 'conflicts': 231, 'rule': 'keep_ponzi', 'contracts': 3340, 'ponzi': 133}
Loaded 3340 contracts (133 Ponzi)
Token-length stats: {'median_tokens': 927, 'p90_tokens': 2732, 'max_tokens': 14700, 'frac_truncated_at_max_len': 0.4578}

=== xgboost | loss=focal | max_len=1024 | fold0 ===
  split: 2404/268/668 (test Ponzi: 26)
  TEST: {'precision': 0.7368, 'recall': 0.5385, 'f1': 0.6222, 'roc_auc': 0.9179, 'pr_auc': 0.6836, 'runtime_sec': 25.4}

=== xgboost | loss=focal | max_len=1024 | fold1 ===
  split: 2404/268/668 (test Ponzi: 26)
  TEST: {'precision': 0.875, 'recall': 0.5385, 'f1': 0.6667, 'roc_auc': 0.9067, 'pr_auc': 0.6516, 'runtime_sec': 23.3}

=== xgboost | loss=focal | max_len=1024 | fold2 ===
  split: 2404/268/668 (test Ponzi: 27)
  TEST: {'precision': 0.5312, 'recall': 0.6296, 'f1': 0.5763, 'roc_auc': 0.9343, 'pr_auc': 0.6149, 'runtime_sec': 33.9}

=== xgboost | loss=focal | max_len=1024 | fold3 ===
  s

  lexical branch: 2000 features
  embedding cache: 4021 contracts loaded (data\emb_microsoft_graphcodebert_base_4x512.npz)
  probe heads trained: 3 model(s), 258K params each, 43.6s
  TEST: {'precision': 0.7083, 'recall': 0.6538, 'f1': 0.68, 'roc_auc': 0.9403, 'pr_auc': 0.751, 'threshold': 0.19, 'runtime_sec': 61.2}

=== transfer | loss=bce | max_len=512 | fold1 ===
  split: 2404/268/668 (test Ponzi: 26)
  device: cuda
  loading microsoft/graphcodebert-base (downloads once, ~500 MB)...
  lexical branch: 2000 features
  embedding cache: 4021 contracts (in memory)
  probe heads trained: 3 model(s), 258K params each, 61.4s
  TEST: {'precision': 0.4839, 'recall': 0.5769, 'f1': 0.5263, 'roc_auc': 0.9098, 'pr_auc': 0.6139, 'threshold': 0.0768, 'runtime_sec': 66.0}

=== transfer | loss=bce | max_len=512 | fold2 ===
  split: 2404/268/668 (test Ponzi: 27)
  device: cuda
  loading microsoft/graphcodebert-base (downloads once, ~500 MB)...
  lexical branch: 2000 features
  embedding cache: 4021 co

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    2106 non-empty chunks to encode (skipped 238 empty)
    1920/2106 chunks (177/s, ~0.0 min left)   
  embeddings cached (13s)
  probe heads trained: 3 model(s), 258K params each, 111.0s
  TEST: {'precision': 0.775, 'recall': 0.7949, 'f1': 0.7848, 'roc_auc': 0.9749, 'pr_auc': 0.8501, 'threshold': 0.4814, 'runtime_sec': 136.0}

=== transfer | loss=bce | max_len=512 | fold1 ===
  split: 3316/369/922 (test Ponzi: 39)
  device: cuda
  loading microsoft/graphcodebert-base (downloads once, ~500 MB)...
  lexical branch: 2000 features
  embedding cache: 4607 contracts (in memory)
  probe heads trained: 3 model(s), 258K params each, 65.1s
  TEST: {'precision': 0.7692, 'recall': 0.7692, 'f1': 0.7692, 'roc_auc': 0.9875, 'pr_auc': 0.8523, 'threshold': 0.2501, 'runtime_sec': 71.1}

=== transfer | loss=bce | max_len=512 | fold2 ===
  split: 3317/369/921 (test Ponzi: 38)
  device: cuda
  loading microsoft/graphcodebert-base (downloads once, ~500 MB)...
  lexical branch: 2000 features
  embedding ca

In [37]:
# --------------------------------------------------------------- B2
# Conflict-resolution-rule sensitivity  ->  manuscript Table 7
rows = []
try:
    for rule in ("keep_ponzi", "drop_conflict", "majority", "random"):
        st = build_benchmark(jaccard_thr=0.90, rule=rule, seed=0)
        tagx, tagp = f"sens-rule-{rule}-xgb", f"sens-rule-{rule}-probe"
        sx = run_experiment(model="xgboost", cv=5, max_len=1024, tag=tagx)
        sp = run_experiment(model="transfer", cv=5, loss="bce",
                            batch_size=64, lr=1e-3, epochs=40, patience=8,
                            n_models=3, time_budget_min=30, tag=tagp)
        m, s, ff = fuse_fixed_from_tags(tagp, tagx)
        rows.append((rule, st["contracts"], st["ponzi"],
                     f"{sx['f1']['mean']:.3f} ± {sx['f1']['std']:.3f}",
                     f"{m:.3f} ± {s:.3f}"))
finally:
    restore_strict()
print("\n=== Table 7 rows (rule | contracts | Ponzi | XGBoost F1 | "
      "LNF F1) ===")
for r in rows:
    print(" | ".join(str(x) for x in r))



benchmark: {'thr': 0.9, 'pairs': 14648, 'conflicts': 113, 'rule': 'keep_ponzi', 'contracts': 4021, 'ponzi': 158}
Loaded 4021 contracts (158 Ponzi)
Token-length stats: {'median_tokens': 917, 'p90_tokens': 2691, 'max_tokens': 14700, 'frac_truncated_at_max_len': 0.4499}

=== xgboost | loss=focal | max_len=1024 | fold0 ===
  split: 2894/322/805 (test Ponzi: 32)
  TEST: {'precision': 0.9333, 'recall': 0.4375, 'f1': 0.5957, 'roc_auc': 0.9391, 'pr_auc': 0.8012, 'runtime_sec': 28.1}

=== xgboost | loss=focal | max_len=1024 | fold1 ===
  split: 2895/322/804 (test Ponzi: 31)
  TEST: {'precision': 0.9524, 'recall': 0.6452, 'f1': 0.7692, 'roc_auc': 0.9554, 'pr_auc': 0.7722, 'runtime_sec': 27.1}

=== xgboost | loss=focal | max_len=1024 | fold2 ===
  split: 2895/322/804 (test Ponzi: 31)
  TEST: {'precision': 0.9375, 'recall': 0.4839, 'f1': 0.6383, 'roc_auc': 0.9523, 'pr_auc': 0.7121, 'runtime_sec': 25.2}

=== xgboost | loss=focal | max_len=1024 | fold3 ===
  split: 2895/322/804 (test Ponzi: 32)
  TE

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    16 non-empty chunks to encode (skipped 0 empty)
    16/16 chunks (39/s, ~0.0 min left)   
  embeddings cached (2s)
  probe heads trained: 3 model(s), 258K params each, 65.1s
  TEST: {'precision': 0.8, 'recall': 0.5161, 'f1': 0.6275, 'roc_auc': 0.9379, 'pr_auc': 0.6975, 'threshold': 0.6555, 'runtime_sec': 73.8}

=== transfer | loss=bce | max_len=512 | fold1 ===
  split: 2895/322/804 (test Ponzi: 30)
  device: cuda
  loading microsoft/graphcodebert-base (downloads once, ~500 MB)...
  lexical branch: 2000 features
  embedding cache: 4611 contracts (in memory)
  probe heads trained: 3 model(s), 258K params each, 75.1s
  TEST: {'precision': 0.8571, 'recall': 0.6, 'f1': 0.7059, 'roc_auc': 0.9513, 'pr_auc': 0.7415, 'threshold': 0.289, 'runtime_sec': 79.9}

=== transfer | loss=bce | max_len=512 | fold2 ===
  split: 2895/322/804 (test Ponzi: 30)
  device: cuda
  loading microsoft/graphcodebert-base (downloads once, ~500 MB)...
  lexical branch: 2000 features
  embedding cache: 4611 contract

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    1100 non-empty chunks to encode (skipped 48 empty)
    640/1100 chunks (167/s, ~0.0 min left)   
  embeddings cached (8s)
  probe heads trained: 3 model(s), 258K params each, 49.7s
  TEST: {'precision': 0.8276, 'recall': 0.7742, 'f1': 0.8, 'roc_auc': 0.9536, 'pr_auc': 0.7764, 'threshold': 0.1209, 'runtime_sec': 66.4}

=== transfer | loss=bce | max_len=512 | fold1 ===
  split: 2895/322/804 (test Ponzi: 30)
  device: cuda
  loading microsoft/graphcodebert-base (downloads once, ~500 MB)...
  lexical branch: 2000 features
  embedding cache: 4898 contracts (in memory)
  probe heads trained: 3 model(s), 258K params each, 46.1s
  TEST: {'precision': 0.8333, 'recall': 0.5, 'f1': 0.625, 'roc_auc': 0.8639, 'pr_auc': 0.5798, 'threshold': 0.173, 'runtime_sec': 51.0}

=== transfer | loss=bce | max_len=512 | fold2 ===
  split: 2895/322/804 (test Ponzi: 30)
  device: cuda
  loading microsoft/graphcodebert-base (downloads once, ~500 MB)...
  lexical branch: 2000 features
  embedding cache: 4898 co

In [38]:
# --------------------------------------------------------------- B3
# Per-model leakage quantification  ->  manuscript Table 8
# Conventional protocol = exact deduplication only (near-duplicates
# kept), random stratified splits - the identical pipelines otherwise.
# Runtime: the neural runs dominate (~2-4 h total with the budgets
# below). If GPU memory is tight at max_len=1024, keep the reduced
# batch sizes given here; restart the kernel first to clear leftover
# allocations.
import torch
conv = {}
try:
    build_benchmark(rule="keep_near_dups")        # 5,917 contracts
    conv["TF–IDF + XGBoost"] = run_experiment(
        model="xgboost", cv=5, max_len=1024, tag="conv-xgb")
    torch.cuda.empty_cache()
    conv["TextCNN"] = run_experiment(
        model="textcnn", cv=5, max_len=1024, tag="conv-textcnn")
    torch.cuda.empty_cache()
    conv["Transformer (L=1,024)"] = run_experiment(
        model="transformer", cv=5, max_len=1024, batch_size=16,
        tag="conv-transformer")
    torch.cuda.empty_cache()
    conv["PonziMamba SSM"] = run_experiment(
        model="mamba", cv=5, max_len=1024, loss="bce", batch_size=16,
        tag="conv-mamba")
    torch.cuda.empty_cache()
    conv["Frozen probe (K=4)"] = run_experiment(
        model="transfer", cv=5, loss="bce", batch_size=64, lr=1e-3,
        epochs=40, patience=8, n_models=3, time_budget_min=30,
        tag="conv-probe")
    m, s, ff = fuse_fixed_from_tags("conv-probe", "conv-xgb")
    conv["LNF (proposed)"] = {"f1": {"mean": m, "std": s}}
finally:
    restore_strict()

STRICT = {"TF–IDF + XGBoost": (0.672, 0.047),
          "TextCNN": (0.662, 0.051),
          "Transformer (L=1,024)": (0.587, 0.055),
          "PonziMamba SSM": (0.695, 0.068),
          "Frozen probe (K=4)": (0.688, 0.066),
          "LNF (proposed)": (0.738, 0.044)}
print("\n=== Table 8 rows (model | conventional F1 | strict F1 | "
      "ΔF1) ===")
for name, summ in conv.items():
    cm = summ["f1"]["mean"]
    sm, ss = STRICT[name]
    print(f"{name} | {cm:.3f} ± {summ['f1']['std']:.3f} | "
          f"{sm:.3f} ± {ss:.3f} | {cm - sm:+.3f}")


benchmark: {'thr': None, 'pairs': None, 'conflicts': None, 'rule': 'keep_near_dups', 'contracts': 5917, 'ponzi': 260}
Loaded 5917 contracts (260 Ponzi)
Token-length stats: {'median_tokens': 969, 'p90_tokens': 2853, 'max_tokens': 14700, 'frac_truncated_at_max_len': 0.4786}

=== xgboost | loss=focal | max_len=1024 | fold0 ===
  split: 4259/474/1184 (test Ponzi: 52)
  TEST: {'precision': 0.8571, 'recall': 0.6923, 'f1': 0.766, 'roc_auc': 0.9812, 'pr_auc': 0.8528, 'runtime_sec': 31.2}

=== xgboost | loss=focal | max_len=1024 | fold1 ===
  split: 4259/474/1184 (test Ponzi: 52)
  TEST: {'precision': 0.8, 'recall': 0.8462, 'f1': 0.8224, 'roc_auc': 0.9899, 'pr_auc': 0.8966, 'runtime_sec': 38.0}

=== xgboost | loss=focal | max_len=1024 | fold2 ===
  split: 4260/474/1183 (test Ponzi: 52)
  TEST: {'precision': 0.8235, 'recall': 0.8077, 'f1': 0.8155, 'roc_auc': 0.9842, 'pr_auc': 0.8335, 'runtime_sec': 33.0}

=== xgboost | loss=focal | max_len=1024 | fold3 ===
  split: 4260/474/1183 (test Ponzi: 52)

C:\Users\Omar\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  parameters: 1.81M
    first batch done - est. 1.2 min/epoch (267 batches); worst case this fold ~35 min
  epoch 01 loss=0.0184 val_f1=0.6667 val_auc=0.9163 (17.1s)
  epoch 02 loss=0.0095 val_f1=0.6842 val_auc=0.9019 (17.0s)
  epoch 03 loss=0.0049 val_f1=0.7027 val_auc=0.8989 (17.1s)
  epoch 04 loss=0.0024 val_f1=0.6500 val_auc=0.8921 (17.0s)
  epoch 05 loss=0.0014 val_f1=0.6500 val_auc=0.8950 (16.8s)
  epoch 06 loss=0.0009 val_f1=0.6667 val_auc=0.8880 (16.9s)
  epoch 07 loss=0.0010 val_f1=0.6667 val_auc=0.8859 (16.9s)
  epoch 08 loss=0.0006 val_f1=0.6500 val_auc=0.8834 (16.8s)
  epoch 09 loss=0.0007 val_f1=0.6500 val_auc=0.8817 (16.7s)
  early stop
  TEST: {'precision': 0.8611, 'recall': 0.5962, 'f1': 0.7045, 'roc_auc': 0.9775, 'pr_auc': 0.7545, 'threshold': 0.8708, 'peak_gpu_mem_mb': 604.5, 'runtime_sec': 157.7}

=== transformer | loss=focal | max_len=1024 | fold1 ===
  split: 4259/474/1184 (test Ponzi: 52)
  device: cuda
  vocab size: 10002


C:\Users\Omar\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  parameters: 1.81M
    first batch done - est. 1.7 min/epoch (267 batches); worst case this fold ~50 min
  epoch 01 loss=0.0205 val_f1=0.7222 val_auc=0.9481 (17.2s)
  epoch 02 loss=0.0112 val_f1=0.7429 val_auc=0.9872 (16.9s)
  epoch 03 loss=0.0060 val_f1=0.7692 val_auc=0.9854 (16.9s)
  epoch 04 loss=0.0032 val_f1=0.8421 val_auc=0.9714 (17.0s)
  epoch 05 loss=0.0017 val_f1=0.8649 val_auc=0.9892 (17.0s)
  epoch 06 loss=0.0014 val_f1=0.8205 val_auc=0.9845 (17.0s)
  epoch 07 loss=0.0011 val_f1=0.8649 val_auc=0.9860 (16.7s)
  epoch 08 loss=0.0008 val_f1=0.8000 val_auc=0.9892 (17.0s)
  epoch 09 loss=0.0010 val_f1=0.8000 val_auc=0.9785 (16.6s)
  epoch 10 loss=0.0010 val_f1=0.8205 val_auc=0.9854 (16.5s)
  epoch 11 loss=0.0009 val_f1=0.8000 val_auc=0.9663 (16.7s)
  early stop
  TEST: {'precision': 0.8372, 'recall': 0.6923, 'f1': 0.7579, 'roc_auc': 0.9814, 'pr_auc': 0.8189, 'threshold': 0.5693, 'peak_gpu_mem_mb': 604.5, 'runtime_sec': 191.1}

=== transformer | loss=focal | max_len=1024 | fold2 

...

C:\Users\Omar\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  parameters: 1.81M
    first batch done - est. 1.4 min/epoch (267 batches); worst case this fold ~42 min
  epoch 01 loss=0.0195 val_f1=0.6857 val_auc=0.9583 (16.9s)
  epoch 02 loss=0.0105 val_f1=0.6857 val_auc=0.9771 (16.8s)
  epoch 03 loss=0.0049 val_f1=0.8108 val_auc=0.9841 (17.0s)
  epoch 04 loss=0.0026 val_f1=0.7429 val_auc=0.9745 (17.1s)
  epoch 05 loss=0.0017 val_f1=0.8108 val_auc=0.9865 (16.7s)
  epoch 06 loss=0.0013 val_f1=0.8000 val_auc=0.9902 (16.7s)
  epoch 07 loss=0.0011 val_f1=0.7692 val_auc=0.9811 (17.0s)
  epoch 08 loss=0.0012 val_f1=0.7692 val_auc=0.9896 (17.3s)
  epoch 09 loss=0.0008 val_f1=0.7429 val_auc=0.9880 (17.2s)
  early stop
  TEST: {'precision': 0.75, 'recall': 0.75, 'f1': 0.75, 'roc_auc': 0.9557, 'pr_auc': 0.8168, 'threshold': 0.5587, 'peak_gpu_mem_mb': 604.5, 'runtime_sec': 157.7}

================ SUMMARY ================
{
  "precision": {
    "mean": 0.8072,
    "std": 0.0515
  },
  "recall": {
    "mean": 0.7,
    "std": 0.0565
  },
  "f1": {
    "mean"

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    4003 non-empty chunks to encode (skipped 73 empty)
    3840/4003 chunks (183/s, ~0.0 min left)   
  embeddings cached (24s)
  probe heads trained: 3 model(s), 258K params each, 62.9s
  TEST: {'precision': 0.8431, 'recall': 0.8269, 'f1': 0.835, 'roc_auc': 0.9825, 'pr_auc': 0.8937, 'threshold': 0.1919, 'runtime_sec': 102.6}

=== transfer | loss=bce | max_len=512 | fold1 ===
  split: 4259/474/1184 (test Ponzi: 52)
  device: cuda
  loading microsoft/graphcodebert-base (downloads once, ~500 MB)...
  lexical branch: 2000 features
  embedding cache: 5917 contracts (in memory)
  probe heads trained: 3 model(s), 258K params each, 93.9s
  TEST: {'precision': 0.7586, 'recall': 0.8462, 'f1': 0.8, 'roc_auc': 0.9756, 'pr_auc': 0.8877, 'threshold': 0.1244, 'runtime_sec': 100.8}

=== transfer | loss=bce | max_len=512 | fold2 ===
  split: 4260/474/1183 (test Ponzi: 52)
  device: cuda
  loading microsoft/graphcodebert-base (downloads once, ~500 MB)...
  lexical branch: 2000 features
  embedding cach

### Transcribing Part B output into the manuscript

- **Table 6**: paste the four `Table 6 rows` lines (threshold, pairs, conflicting,
  contracts, Ponzi, XGBoost F1, LNF F1). The 0.90 row must reproduce
  14,648 / 113 / 4,021 / 158 / 0.672 ± 0.047 / 0.738 ± 0.044 — if it does not,
  stop and check the environment before filling anything.
- **Table 7**: paste the four `Table 7 rows` lines.
- **Table 8**: paste the six `Table 8 rows` lines (the XGBoost row should land near
  the 0.820 → 0.672 drop already quoted in Section 4).
- Then update the one sentence in Section 4 and the Section 6.6 paragraph **only if**
  a measured number contradicts the text (e.g., if detector ranking changes under
  some rule, say so rather than keeping the “ranking unchanged” phrasing).